In [4]:
# ============================================================
# Cell 1: Imports & Global Configuration
# ============================================================

# ── Standard library ────────────────────────────────────────
import os

# ── Numerics & plotting ─────────────────────────────────────
import numpy as np
import matplotlib
matplotlib.use('Agg')                        # non-interactive backend (no display needed)
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# ── Astropy ─────────────────────────────────────────────────
from astropy.cosmology import Planck18 as cosmo
from astropy import constants as const

# ── 21cmFAST ────────────────────────────────────────────────
try:
    import py21cmfast as p21
except Exception as e:
    raise RuntimeError("py21cmfast is required. Install via: pip install py21cmfast") from e

# ============================================================
# Plot Style
# ============================================================
plt.rcParams.update({
    # Fonts
    'font.family'       : 'serif',
    'font.serif'        : ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset'  : 'cm',
    'font.size'         : 20,
    'axes.labelsize'    : 20,
    'axes.titlesize'    : 20,
    'xtick.labelsize'   : 14,
    'ytick.labelsize'   : 14,
    'legend.fontsize'   : 14,
    'figure.titlesize'  : 20,
    # Ticks — inside, all four sides
    'xtick.direction'   : 'in',
    'ytick.direction'   : 'in',
    'xtick.top'         : True,
    'ytick.right'       : True,
    'xtick.major.size'  : 6,   'ytick.major.size'  : 6,
    'xtick.minor.size'  : 3,   'ytick.minor.size'  : 3,
    'xtick.major.width' : 1.0, 'ytick.major.width' : 1.0,
    'xtick.minor.width' : 0.8, 'ytick.minor.width' : 0.8,
    # Axes & lines
    'axes.linewidth'    : 1.2,
    'lines.linewidth'   : 2.0,
    'lines.markersize'  : 5,
    # No grid
    'axes.grid'         : False,
    # Output
    'figure.dpi'        : 150,
    'savefig.dpi'       : 300,
    'savefig.bbox'      : 'tight',
    'savefig.pad_inches': 0.05,
})
mpl.rcParams['xtick.minor.visible'] = True
mpl.rcParams['ytick.minor.visible'] = True

# ============================================================
# 21cmFAST Runtime Settings
# ============================================================

CACHE_DIR = "28May2026/Cache/"
PLOT_DIR  = "28May2026/Plots/"

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(PLOT_DIR,  exist_ok=True)

p21.config['direc'] = CACHE_DIR

os.environ['OMP_NUM_THREADS'] = '32'

print(f"Cache : {CACHE_DIR}")
print(f"Plots : {PLOT_DIR}")

# ============================================================
# Physical Constants  (cgs throughout)
# ============================================================
sigma_T = const.sigma_T.cgs.value            # Thomson cross-section  [cm²]
MPC_CM  = 3.0856775814913673e24              # 1 Mpc in cm            [cm/Mpc]

# ============================================================
# Simulation Box & Redshift Grid
# ============================================================
HII_DIM = 128                                # grid resolution [cells/side]
BOX_LEN = 800.0                              # comoving box side length [Mpc]
voxel   = BOX_LEN / HII_DIM                 # cell size [Mpc]

ZS = [                                       # redshift slices (decreasing)
    16.0, 15.5, 15.0, 14.5, 14.0,
    13.5, 13.0, 12.5, 12.0, 11.5,
    11.0, 10.5, 10.0,  9.5,  9.0,
     8.5,  8.0,  7.5,  7.0,  6.5,
     6.0,  5.5,  5.0, 4.5, 4.0, 3.5, 3.0, 2.5, 2.0, 1.5, 1.0, 0.5, 0.001
]

BOX_LABEL = f"Box: {BOX_LEN:.0f} Mpc  |  {HII_DIM}³  |  Δx = {voxel:.2f} Mpc"
print(BOX_LABEL)
print("Redshifts:", ZS)

Cache : 28May2026/Cache/
Plots : 28May2026/Plots/
Box: 800 Mpc  |  128³  |  Δx = 6.25 Mpc
Redshifts: [16.0, 15.5, 15.0, 14.5, 14.0, 13.5, 13.0, 12.5, 12.0, 11.5, 11.0, 10.5, 10.0, 9.5, 9.0, 8.5, 8.0, 7.5, 7.0, 6.5, 6.0, 5.5, 5.0, 4.5, 4.0, 3.5, 3.0, 2.5, 2.0, 1.5, 1.0, 0.5, 0.001]


In [39]:
# ── Extended velocity sanity check ───────────────────────────
print(f"\n{'z':>6}  {'vel_fac':>10}  {'v_rms':>10}  {'v_max':>10}  {'v_mode':>10}  (all km/s)")
print("-" * 58)

for z_test in [15.0, 10.0, 7.0, 5.0]:          # skip z=300, unphysical for coeval
    fac = velocity_conversion_factor(z_test)

    coeval = p21.run_coeval(
        redshift     = float(z_test),
        user_params  = {"HII_DIM": 64, "BOX_LEN": float(BOX_LEN)},
        write        = False,
    )
    vx_km = coeval.lowres_vx * fac              # [km/s]

    # ── NaN check ────────────────────────────────────────────
    n_nan = np.isnan(vx_km).sum()
    if n_nan > 0:
        print(f"  z={z_test}: WARNING {n_nan} NaNs found — masking")
        vx_km = vx_km[np.isfinite(vx_km)]

    v_rms  = vx_km.std()
    v_max  = np.abs(vx_km).max()

    # mode of signed distribution (should be ~0 for Gaussian)
    counts, edges = np.histogram(vx_km.ravel(), bins=200)
    v_mode = 0.5 * (edges[np.argmax(counts)] + edges[np.argmax(counts) + 1])

    # mode of |v| distribution (most probable speed)
    counts_abs, edges_abs = np.histogram(np.abs(vx_km.ravel()), bins=200)
    v_mode_abs = 0.5 * (edges_abs[np.argmax(counts_abs)] + edges_abs[np.argmax(counts_abs) + 1])

    print(f"{z_test:>6.1f}  {fac:>10.4f}  {v_rms:>10.2f}  "
          f"{v_max:>10.2f}  {v_mode:>10.3f}  |v|_mode={v_mode_abs:.2f}")


     z     vel_fac       v_rms       v_max      v_mode  (all km/s)
----------------------------------------------------------
  15.0      3.8612       33.10      145.11       2.651  |v|_mode=0.36
  10.0      4.6455       34.88      163.36      -0.085  |v|_mode=0.41
   7.0      5.4419       42.78      194.52       0.754  |v|_mode=1.46
   5.0      6.2857       49.27      215.61       4.707  |v|_mode=4.85


In [5]:
# ============================================================
# Cell 2: Helper Functions
# ============================================================
from scipy.integrate import quad
import astropy.units as u

# ── Growth factor D(z), normalized to 1 at z=0 ──────────────
def growth_factor(z):
    """
    D(z) via the standard integral definition, normalized to 1 at z=0.
    D(z) ∝ H(z) * integral_z^inf  dz' (1+z') / H(z')^3
    """
    def integrand(zp):
        return (1.0 + zp) / cosmo.H(zp).value**3
    val,  _ = quad(integrand, z,   np.inf)
    norm, _ = quad(integrand, 0.0, np.inf)
    return (cosmo.H(z).value * val) / (cosmo.H(0).value * norm)

# ── Growth rate f(z) = dlnD/dlna ────────────────────────────
def growth_rate(z):
    """
    f(z) = dlnD/dlna ≈ Omega_m(z)^0.55  (Linder 2005).
    Accurate to ~1% for flat ΛCDM. Always positive.
    """
    return cosmo.Om(z) ** 0.55

# ── dD/dt [s⁻¹], always positive ────────────────────────────
def Ddot(z):
    """dD/dt = D(z) · f(z) · H(z)  [s⁻¹], always positive."""
    H_si = cosmo.H(z).to(1 / u.s).value
    return growth_factor(z) * growth_rate(z) * H_si

# ── Velocity conversion factor ───────────────────────────────
Z_INIT    = 300.0                  # 21cmFAST default INITIAL_REDSHIFT
DDOT_INIT = Ddot(Z_INIT)          # kept for reference

def velocity_conversion_factor(z):
    """
    Converts lowres_vx from 21cmFAST IC units to COMOVING peculiar
    velocity at redshift z [km/s].

    Convention matches Cain+2024 Eq.(2) / Park+2013 Eq.(A16):
        q(x,z) = (1+δ) · χ · v_comoving
        v_comoving = v_physical / (1+z) = D(z)·f(z)·H(z)·Ψ / (1+z)

    The ds/a⁴ factor in the C_ell integral (not ds/a⁵) confirms
    that v in q must be the COMOVING peculiar velocity.

    Gives v_rms ~ 95-156 km/s at z=5-15 for an 800 Mpc box.
    Caller multiplies by 1e5 to convert km/s → cm/s.
    """
    return (growth_factor(z) * growth_rate(z)
            * cosmo.H(z).value / (1.0 + z))         # [km/s]

# ── Sanity check (empirical, small box for speed) ────────────
print(f"{'z':>6}  {'vel_fac':>14}  {'v_rms':>10}  {'v_max':>10}  (km/s)")
print("-" * 48)
for z_test in [15.0, 10.0, 7.0, 5.0, 1.0, 0.5]:
    fac    = velocity_conversion_factor(z_test)
    coeval = p21.run_coeval(
        redshift     = float(z_test),
        user_params  = {"HII_DIM": 64, "BOX_LEN": float(BOX_LEN)},
        write        = False,
    )
    vx_km = coeval.lowres_vx * fac
    print(f"{z_test:>6.1f}  {fac:>14.4f}  "
          f"{vx_km.std():>10.2f}  "
          f"{np.abs(vx_km).max():>10.2f}")

# ── Mean electron number density today ──────────────────────
def ne0_cgs(Y_He=0.24, include_He=True):
    """
    Mean comoving electron number density today [cm⁻³].
    Assumes fully ionized H and singly ionized He (He II).

    Parameters
    ----------
    Y_He       : helium mass fraction (default 0.24)
    include_He : if True, adds He II electrons to the count
    """
    m_p    = const.m_p.cgs.value
    rho_c0 = cosmo.critical_density0.cgs.value
    X_H    = 1.0 - Y_He
    n_H0   = X_H * (cosmo.Ob0 * rho_c0) / m_p
    if include_He:
        y = Y_He / (4.0 * X_H)          # He/H number ratio
        electrons_per_H = 1.0 + y        # 1 from H + 1 from He II
    else:
        electrons_per_H = 1.0
    return n_H0 * electrons_per_H

ne0 = ne0_cgs()
print(f"\nn_e0 = {ne0:.4e} cm⁻³")

# ── Run a coeval box and extract physical fields ─────────────
def run_coeval_fields(z, HII_DIM=HII_DIM, BOX_LEN=BOX_LEN,
                      user_overrides=None, astro_overrides=None):
    """
    Run a 21cmFAST coeval box at redshift z and return physical fields.

    Returns
    -------
    delta    : matter overdensity δ          (dimensionless)
    xH       : neutral hydrogen fraction     (dimensionless)
    vx/vy/vz : physical peculiar velocities  (cm s⁻¹)

    Notes
    -----
    lowres_vx stores the Zel'dovich displacement Ψ (dimensionless IC
    field). Physical peculiar velocity at z:
        v_phys(z) = Ψ * D(z) * f(z) * H(z) * 1e5   [cm/s]
    Validated to ~1% against linear theory at z=0.5-15.
    v_phys decreases toward low z (correct ΛCDM: H drops faster than D*f grows).
    """
    user_params  = {"HII_DIM": int(HII_DIM), "BOX_LEN": float(BOX_LEN)}
    astro_params = {}
    if user_overrides:
        user_params.update(user_overrides)
    if astro_overrides:
        astro_params.update(astro_overrides)

    coeval = p21.run_coeval(
        redshift     = float(z),
        user_params  = user_params,
        astro_params = astro_params,
        write        = False,
    )

    fac   = velocity_conversion_factor(z)        # D(z)*f(z)*H(z)  [km/s]
    delta = coeval.density                        # δ               [dimensionless]
    xH    = coeval.xH_box                         # neutral fraction [dimensionless]
    vx    = coeval.lowres_vx * fac * 1e5          # v_phys          [cm s⁻¹]
    vy    = coeval.lowres_vy * fac * 1e5
    vz    = coeval.lowres_vz * fac * 1e5

    return delta, xH, vx, vy, vz

# ── Build ionized momentum field ─────────────────────────────
def build_momentum(delta, xH, vx, vy, vz, z):
    """
    Ionized electron momentum field  q = n_e · v  [cm⁻² s⁻¹].

    n_e(x,z) = ne0 · (1+z)³ · (1+δ) · χ,   χ = 1 - xH
    vx/vy/vz must already be physical peculiar velocities in cm s⁻¹.
    """
    chi      = 1.0 - xH
    ne_fluct = ne0 * (1.0 + z)**3 * (1.0 + delta) * chi    # [cm⁻³]
    qx = ne_fluct * vx    # [cm⁻² s⁻¹]
    qy = ne_fluct * vy
    qz = ne_fluct * vz
    return qx, qy, qz

     z         vel_fac       v_rms       v_max  (km/s)
------------------------------------------------


/home/swanith/miniconda3/envs/cmfast_tmp/lib/python3.10/site-packages/py21cmfast/inputs.py:515: UserWarning: The USE_INTERPOLATION_TABLES setting has changed in v3.1.2 to be default True. You can likely ignore this warning, but if you relied onhaving USE_INTERPOLATION_TABLES=False by *default*, please set it explicitly. To silence this warning, set it explicitly to True. Thiswarning will be removed in v4.
  warnings.warn(


  15.0         11.8833       93.78      397.24
  10.0         14.3666      113.64      456.74
   7.0         16.8631      132.32      570.80
   5.0         19.4660      150.24      676.06
   1.0         31.9603      257.81     1235.51
   0.5         34.6072      268.51     1264.02

n_e0 = 2.0644e-07 cm⁻³


In [7]:
# ============================================================
# Cell 3: Power Spectra of δ, xH, v, and δ_e fields
# ============================================================
import pickle, os

# ── Power spectrum engine ────────────────────────────────────
def compute_power_spectrum(field, BOX_LEN=BOX_LEN):
    """
    Spherically averaged 3D power spectrum of a real field.
    Returns k [Mpc⁻¹] and P(k) [Mpc³ · field_unit²].
    """
    N    = field.shape[0]
    dk   = 2.0 * np.pi / BOX_LEN

    fft  = np.fft.fftshift(np.fft.fftn(field))
    pk3d = np.abs(fft)**2 * (BOX_LEN / N**2)**3

    ki         = np.fft.fftshift(np.fft.fftfreq(N, d=1.0/N))
    kx, ky, kz = np.meshgrid(ki, ki, ki, indexing='ij')
    k_mag      = np.sqrt(kx**2 + ky**2 + kz**2) * dk

    k_bins  = np.arange(0.5, N//2 + 1, 1.0) * dk
    k_mids  = 0.5 * (k_bins[:-1] + k_bins[1:])
    P_mean  = np.zeros(len(k_mids))
    N_modes = np.zeros(len(k_mids), dtype=int)

    for i, (klo, khi) in enumerate(zip(k_bins[:-1], k_bins[1:])):
        mask       = (k_mag >= klo) & (k_mag < khi)
        N_modes[i] = mask.sum()
        if N_modes[i] > 0:
            P_mean[i] = pk3d[mask].mean()

    good = N_modes > 0
    return k_mids[good], P_mean[good]

def compute_velocity_power(vx, vy, vz, BOX_LEN=BOX_LEN):
    """
    Isotropic velocity power spectrum: P_vv = (Pxx + Pyy + Pzz) / 3.
    """
    k, Px = compute_power_spectrum(vx, BOX_LEN)
    _, Py = compute_power_spectrum(vy, BOX_LEN)
    _, Pz = compute_power_spectrum(vz, BOX_LEN)
    return k, (Px + Py + Pz) / 3.0

def compute_electron_overdensity(delta, xH):
    """
    Free electron overdensity field δ_e(x,z).

    n_e(x,z) = n̄_e(z) · (1+δ) · χ,   χ = 1 - xH
    δ_e      = (1+δ)·χ / <(1+δ)·χ>  - 1

    Returns δ_e and the volume-averaged ionisation fraction x_e.
    """
    chi      = 1.0 - xH
    ne_field = (1.0 + delta) * chi
    ne_mean  = ne_field.mean()
    delta_e  = ne_field / ne_mean - 1.0
    return delta_e, float(ne_mean)

# ── Cache helpers ────────────────────────────────────────────
CACHE_PS = os.path.join(CACHE_DIR, "power_spectra.pkl")

def save_ps_cache(results):
    with open(CACHE_PS, 'wb') as f:
        pickle.dump(results, f)
    print(f"Cache saved → {CACHE_PS}")

def load_ps_cache():
    if os.path.exists(CACHE_PS):
        with open(CACHE_PS, 'rb') as f:
            results = pickle.load(f)
        print(f"Cache loaded ← {CACHE_PS}  ({len(results)} redshifts)")
        return results
    return None

# ── Redshift grid ────────────────────────────────────────────
ZS_PS       = list(np.arange(15.0, 0.0, -0.5))   # 15.0 → 0.5
FORCE_RERUN = False    # old cache had wrong velocity conversion — must rerun

results_ps = None if FORCE_RERUN else load_ps_cache()
if results_ps is None:
    results_ps = {}

missing = [z for z in ZS_PS if z not in results_ps]
if missing:
    print(f"Computing {len(missing)} redshifts...\n")
else:
    print("Cache covers all redshifts — skipping computation.")

for z in missing:
    print(f"  z={z:.1f}...", end=' ', flush=True)

    delta, xH, vx, vy, vz = run_coeval_fields(z)
    vx_km = vx / 1e5
    vy_km = vy / 1e5
    vz_km = vz / 1e5

    delta_e, xe = compute_electron_overdensity(delta, xH)

    k_d,  Pd  = compute_power_spectrum(delta,   BOX_LEN)
    k_xH, PxH = compute_power_spectrum(xH,      BOX_LEN)
    k_v,  Pv  = compute_velocity_power(vx_km, vy_km, vz_km, BOX_LEN)
    k_ee, Pee = compute_power_spectrum(delta_e, BOX_LEN)

    results_ps[z] = {
        'k_d'      : k_d,   'Pd'  : Pd,
        'k_xH'     : k_xH,  'PxH' : PxH,
        'k_v'      : k_v,   'Pv'  : Pv,
        'k_ee'     : k_ee,  'Pee' : Pee,
        'xH_mean'  : float(xH.mean()),
        'xe_mean'  : xe,
        'v_rms_km' : float(vx_km.std()),
    }
    print(f"<xH>={xH.mean():.3f}  xe={xe:.3f}  v_rms={vx_km.std():.1f} km/s")

if missing:
    save_ps_cache(results_ps)
    print("\nAll redshifts done.")

# ── Plot — 2×2 + colorbar ────────────────────────────────────
ZS_sorted = sorted(results_ps.keys(), reverse=True)
colors    = plt.cm.plasma(np.linspace(0.05, 0.95, len(ZS_sorted)))

fig = plt.figure(figsize=(16, 11))
fig.suptitle(f"Field Power Spectra  —  {BOX_LABEL}", y=1.01)

gs = fig.add_gridspec(
    2, 3,
    width_ratios = [1, 1, 0.06],
    hspace       = 0.32,
    wspace       = 0.28,
)

ax_d  = fig.add_subplot(gs[0, 0])
ax_xH = fig.add_subplot(gs[0, 1])
ax_v  = fig.add_subplot(gs[1, 0])
ax_ee = fig.add_subplot(gs[1, 1])
ax_cb = fig.add_subplot(gs[:, 2])

for ax in [ax_d, ax_xH, ax_v, ax_ee]:
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'$k\ [\mathrm{Mpc}^{-1}]$')

ax_d.set_ylabel(r'$\Delta^2_\delta(k)$')
ax_d.set_title(r'Matter overdensity $\delta$')

ax_xH.set_ylabel(r'$\Delta^2_{x_\mathrm{H}}(k)$')
ax_xH.set_title(r'Neutral fraction $x_\mathrm{H}$')

ax_v.set_ylabel(r'$P_{vv}(k)\ [\mathrm{km}^2\,\mathrm{s}^{-2}\,\mathrm{Mpc}^3]$')
ax_v.set_title(r'Velocity $v$')

ax_ee.set_ylabel(r'$P_{ee}(k)\ [\mathrm{Mpc}^3]$')
ax_ee.set_title(r'Electron overdensity $\delta_e$')

for z, col in zip(ZS_sorted, colors):
    res      = results_ps[z]
    eor      = z >= 5.0
    alpha    = 1.0 if eor else 0.3
    k        = res['k_d']
    Delta2_d  = k**3 * res['Pd']  / (2.0 * np.pi**2)
    Delta2_xH = k**3 * res['PxH'] / (2.0 * np.pi**2)

    ax_d.plot( res['k_d'],  Delta2_d,   color=col)
    ax_xH.plot(res['k_xH'], Delta2_xH,  color=col, alpha=alpha)
    ax_v.plot( res['k_v'],  res['Pv'],  color=col)
    ax_ee.plot(res['k_ee'], res['Pee'], color=col, alpha=alpha)

# ── Colorbar ─────────────────────────────────────────────────
norm = plt.Normalize(vmin=min(ZS_sorted), vmax=max(ZS_sorted))
sm   = plt.cm.ScalarMappable(cmap='plasma', norm=norm)
sm.set_array([])
cb   = fig.colorbar(sm, cax=ax_cb)
cb.set_label(r'Redshift $z$', labelpad=10)
cb.ax.yaxis.set_ticks_position('right')
cb.ax.yaxis.set_label_position('right')

# ── Save ─────────────────────────────────────────────────────
stem = os.path.join(PLOT_DIR, "power_spectra_delta_xH_v_ee")
plt.savefig(stem + ".pdf")
plt.savefig(stem + ".png", dpi=300)
print(f"Saved → {stem}.pdf")
print(f"Saved → {stem}.png")
plt.close()

Cache loaded ← 28May2026/Cache/power_spectra.pkl  (30 redshifts)
Cache covers all redshifts — skipping computation.
Saved → 28May2026/Plots/power_spectra_delta_xH_v_ee.pdf
Saved → 28May2026/Plots/power_spectra_delta_xH_v_ee.png


In [58]:
# ── P_vv: 21cmFAST vs Linear Theory ──────────────────────────
from scipy.interpolate import interp1d

h        = cosmo.H(0).value / 100.0
colors_z = plt.cm.plasma(np.linspace(0.15, 0.85, 4))
zs_check = [15.0, 10.0, 7.0, 5.0]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(r'$P_{vv}(k)$ — 21cmFAST vs Linear Theory')

ax_p, ax_r = axes
for ax in axes:
    ax.set_xscale('log')
    ax.set_xlabel(r'$k\ [\mathrm{Mpc}^{-1}]$')

ax_p.set_yscale('log')
ax_p.set_ylabel(r'$P_{vv}(k)\ [\mathrm{km}^2\,\mathrm{s}^{-2}\,\mathrm{Mpc}^3]$')
ax_p.set_title(r'$P_{vv}(k)$ comparison')

ax_r.set_ylabel('Ratio: 21cmFAST / Linear theory')
ax_r.set_title('Ratio (expect ~1 on large scales)')
ax_r.axhline(1.0, color='k', ls=':', lw=1.5, label='Perfect agreement')
ax_r.axhline(1.1, color='k', ls='--', lw=0.8, alpha=0.4)
ax_r.axhline(0.9, color='k', ls='--', lw=0.8, alpha=0.4)

for z_test, col in zip(zs_check, colors_z):
    label  = rf'$z={z_test:.0f}$'
    k_sim  = results_ps[z_test]['k_v']
    Pv_sim = results_ps[z_test]['Pv']

    # Linear theory P_vv = (f*H/k)² * P_δδ(k,z)
    Dz     = growth_factor(z_test)
    D0     = growth_factor(0.0)
    f_z    = growth_rate(z_test)
    H_z    = cosmo.H(z_test).value
    pk_z   = pk_lin * (Dz / D0)**2
    Pvv_lt = (f_z * H_z / k_lin)**2 * pk_z

    Pvv_at_ksim = interp1d(k_lin, Pvv_lt,
                           bounds_error=False,
                           fill_value=np.nan)(k_sim)
    ratio = Pv_sim / Pvv_at_ksim

    ax_p.plot(k_sim, Pv_sim,   color=col, lw=2.0,       label=f'Sim {label}')
    ax_p.plot(k_lin, Pvv_lt,   color=col, lw=1.5, ls='--', label=f'LT {label}')
    ax_r.plot(k_sim, ratio,    color=col, lw=2.0,       label=label)

ax_p.legend(frameon=False, ncol=2, fontsize=10)
ax_r.legend(frameon=False)
ax_r.set_ylim(0.0, 2.0)

plt.tight_layout()
stem = os.path.join(PLOT_DIR, "pvv_vs_linear_theory_corrected")
plt.savefig(stem + ".pdf")
plt.savefig(stem + ".png", dpi=300)
print(f"Saved → {stem}.pdf / .png")
plt.close()

Saved → 27April2026/Plots/pvv_vs_linear_theory_corrected.pdf / .png


In [62]:
# ── LoReLi II corrected digitized bounds (k in Mpc⁻¹, P in Mpc³) ──
loreli_lower_x = np.array([
    0.021977, 0.022778, 0.025664, 0.028916, 0.032194, 0.034998,
    0.039433, 0.047726, 0.053136, 0.061315, 0.068265, 0.077838,
    0.086661, 0.096484, 0.107421, 0.123955, 0.148247, 0.163093,
    0.190455, 0.219770, 0.244681, 0.265993, 0.329712, 0.394326,
    0.488786, 0.650836, 0.751013, 0.919879, 1.074207, 1.210324,
    1.380055, 1.554926, 1.837590, 1.904552,
])
loreli_lower_y = np.array([
    5531.681, 11406.249,  7196.857,  8208.914,  6309.573,  6738.627,
    6738.627,  6738.627,  5531.681,  3727.594,  3059.950,  2511.886,
    2351.953,  1692.667,  1389.495,  1068.000,   630.957,   553.168,
     517.947,   372.759,   180.777,   130.103,    67.386,    32.680,
      19.307,    8.767,     3.981,     2.202,     1.693,     0.720,
       0.398,    0.206,     0.107,     0.059,
])

loreli_upper_x = np.array([
    0.021459, 0.025059, 0.031062, 0.039906, 0.057079, 0.074211,
    0.100000, 0.133153, 0.175196, 0.204588, 0.262839, 0.367085,
    0.488786, 0.716015, 0.976421, 1.210324, 1.554926, 1.881964,
])
loreli_upper_y = np.array([
    251188.643, 114062.492, 82089.142, 51794.747, 25118.864, 12181.879,
      5179.475,   2511.886,  1301.025,   820.891,   305.995,   121.819,
        67.386,    20.620,     7.686,     3.728,     1.808,     0.936,
])

# Sort by x
for lx, ly in [(loreli_lower_x, loreli_lower_y),
               (loreli_upper_x, loreli_upper_y)]:
    idx = np.argsort(lx)
    lx[:] = lx[idx];  ly[:] = ly[idx]

# Common k grid for fill_between
k_min    = max(loreli_lower_x.min(), loreli_upper_x.min())
k_max    = min(loreli_lower_x.max(), loreli_upper_x.max())
k_loreli = np.logspace(np.log10(k_min), np.log10(k_max), 500)

y_lo_interp = interp1d(loreli_lower_x, loreli_lower_y,
                        kind='linear')(k_loreli)
y_up_interp = interp1d(loreli_upper_x, loreli_upper_y,
                        kind='linear')(k_loreli)

print(f"LoReLi k range : {k_loreli.min():.4f} — {k_loreli.max():.4f} Mpc⁻¹")
print(f"21cmFAST k range: "
      f"{results_ps[7.0]['k_ee'].min():.4f} — "
      f"{results_ps[7.0]['k_ee'].max():.4f} Mpc⁻¹")

# ── Plot ─────────────────────────────────────────────────────
ZS_sorted  = sorted(results_ps.keys(), reverse=True)
colors_all = plt.cm.plasma(np.linspace(0.05, 0.95, len(ZS_sorted)))

fig, ax = plt.subplots(figsize=(9, 7))
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel(r'$k\ [\mathrm{Mpc}^{-1}]$')
ax.set_ylabel(r'$P_{ee}(k)\ [\mathrm{Mpc}^3]$')
ax.set_title(r'Electron overdensity $P_{ee}(k)$ — 21cmFAST vs LoReLi II')

# ── LoReLi shaded band ────────────────────────────────────────
ax.fill_between(k_loreli, y_lo_interp, y_up_interp,
                color='steelblue', alpha=0.25, zorder=2,
                label='LoReLi II range')
ax.plot(k_loreli, y_lo_interp, color='steelblue', lw=1.2, ls='--', alpha=0.7)
ax.plot(k_loreli, y_up_interp, color='steelblue', lw=1.2, ls='--', alpha=0.7)

# ── 21cmFAST P_ee ─────────────────────────────────────────────
for z, col in zip(ZS_sorted, colors_all):
    res   = results_ps[z]
    eor   = z >= 5.0
    alpha = 1.0 if eor else 0.3
    ax.plot(res['k_ee'], res['Pee'], color=col, lw=1.5, alpha=alpha)

# ── Colorbar ──────────────────────────────────────────────────
norm = plt.Normalize(vmin=min(ZS_sorted), vmax=max(ZS_sorted))
sm   = plt.cm.ScalarMappable(cmap='plasma', norm=norm)
sm.set_array([])
cb   = fig.colorbar(sm, ax=ax, pad=0.02)
cb.set_label(r'Redshift $z$', labelpad=10)

ax.legend(frameon=False, fontsize=11)
plt.tight_layout()

stem = os.path.join(PLOT_DIR, "Pee_vs_LoReLi")
plt.savefig(stem + ".pdf")
plt.savefig(stem + ".png", dpi=300)
print(f"Saved → {stem}.pdf / .png")
plt.close()

LoReLi k range : 0.0220 — 1.8820 Mpc⁻¹
21cmFAST k range: 0.0079 — 0.5027 Mpc⁻¹
Saved → 27April2026/Plots/Pee_vs_LoReLi.pdf / .png


#Cell 3: loop over HII_EFF_FACTOR and z, plotting slices
ZS1 = [12.0]                     # redshifts you want
HII_EFF_values = [50.0]#[1.0, 10.0, 50.0]  # low to high efficiency

for z in ZS1:
    for effval in HII_EFF_values:
        # pass HII_EFF_FACTOR into the coeval run
        delta, chi, vx, vy, vz = run_coeval_fields(
            z,
            astro_overrides={"HII_EFF_FACTOR": effval}
        )
        qx, qy, qz = build_momentum(delta, chi, vx, vy, vz, z)
        
        # ========== STATISTICS FOR ALL FIELDS ==========
        print("\n" + "="*70)
        print(f"FIELD STATISTICS: z={z}, HII_EFF_FACTOR={effval}")
        print("="*70)
        
        # Delta (density contrast)
        print(f"δ (density):  min={delta.min():+.3f}, max={delta.max():+.3f}, "
              f"mean={delta.mean():+.3f}, std={delta.std():.3f}")
        
        # Chi (ionization fraction)
        print(f"χ_e (ion):    min={chi.min():.3f}, max={chi.max():.3f}, "
              f"mean={chi.mean():.3f}, std={chi.std():.3f}")
        
        # Velocities
        print(f"v_x [cm/s]:   min={vx.min():.2e}, max={vx.max():.2e}, "
              f"mean={vx.mean():.2e}, std={vx.std():.2e}")
        print(f"v_y [cm/s]:   min={vy.min():.2e}, max={vy.max():.2e}, "
              f"mean={vy.mean():.2e}, std={vy.std():.2e}")
        print(f"v_z [cm/s]:   min={vz.min():.2e}, max={vz.max():.2e}, "
              f"mean={vz.mean():.2e}, std={vz.std():.2e}")
        
        # Velocity magnitude
        v_mag = np.sqrt(vx**2 + vy**2 + vz**2)
        print(f"|v| [cm/s]:   min={v_mag.min():.2e}, max={v_mag.max():.2e}, "
              f"mean={v_mag.mean():.2e}, std={v_mag.std():.2e}")
        
        # Momentum components
        print(f"q_x [cm/s]:   min={qx.min():.2e}, max={qx.max():.2e}, "
              f"mean={qx.mean():.2e}, std={qx.std():.2e}")
        print(f"q_y [cm/s]:   min={qy.min():.2e}, max={qy.max():.2e}, "
              f"mean={qy.mean():.2e}, std={qy.std():.2e}")
        print(f"q_z [cm/s]:   min={qz.min():.2e}, max={qz.max():.2e}, "
              f"mean={qz.mean():.2e}, std={qz.std():.2e}")
        
        # Momentum magnitude
        q_mag = np.sqrt(qx**2 + qy**2 + qz**2)
        print(f"|q| [cm/s]:   min={q_mag.min():.2e}, max={q_mag.max():.2e}, "
              f"mean={q_mag.mean():.2e}, std={q_mag.std():.2e}")
        
        print("="*70 + "\n")
        
        # central slices (use ASCII variable names)
        delta_sl = central_slice(delta)
        chi_sl   = central_slice(chi)
        vx_sl    = central_slice(vx)
        vy_sl    = central_slice(vy)
        vz_sl    = central_slice(vz)
        qx_sl    = central_slice(qx)
        qy_sl    = central_slice(qy)
        qz_sl    = central_slice(qz)
        
        vel_unit_label = "cm s$^{-1}$"

        # safe filename fragment for HII_EFF_FACTOR (no dots)
        effstr = str(effval).replace('.', 'p') if isinstance(effval, float) else str(effval)

        efflabel = f"{effval:.1e}"   # e.g. "3.0e+01"


# === Individual Professional Plots ===

efflabel = f"{effval:.1e}"
effstr = str(effval).replace('.', 'p') if isinstance(effval, float) else str(effval)

# ========== DENSITY FIELD ==========
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(delta_sl, origin='lower', cmap='cividis', vmin=-0.5, vmax=0.5)
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label(r'Density contrast $\delta$', fontsize=12)
cbar.ax.tick_params(labelsize=10)
ax.set_xlabel('x [pixels]', fontsize=12)
ax.set_ylabel('y [pixels]', fontsize=12)
title_obj = ax.set_title(rf'Density field at $z={z}$, HII\_EFF\_FACTOR$={efflabel}$', fontsize=13)
plt.tight_layout()
plt.savefig(f"delta_z{z}_HIIeff_{effstr}.png", dpi=300)
title_obj.set_visible(False)              # Hide title
plt.savefig(f"delta_z{z}_HIIeff_{effstr}.pdf")
title_obj.set_visible(True)              # Show title
plt.show()

# ========== IONIZATION FIELD ==========
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(chi_sl, origin='lower', cmap='magma', vmin=0, vmax=1)
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label(r'Ionized fraction $x_e$', fontsize=12)
cbar.ax.tick_params(labelsize=10)
ax.set_xlabel('x [pixels]', fontsize=12)
ax.set_ylabel('y [pixels]', fontsize=12)
title_obj = ax.set_title(rf'Ionization field at $z={z}$, HII\_EFF\_FACTOR$={efflabel}$', fontsize=13)
plt.tight_layout()
plt.savefig(f"xe_z{z}_HIIeff_{effstr}.png", dpi=300)
title_obj.set_visible(False)              # Hide title
plt.savefig(f"xe_z{z}_HIIeff_{effstr}.pdf")
title_obj.set_visible(True)              # Show title
plt.show()

# ========== VELOCITY X ==========
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(vx_sl/1e5, origin='lower', cmap='magma', 
               vmin=-30, vmax=30)
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label(r'$v_x$ [$10^5$ cm s$^{-1}$]', fontsize=16)
cbar.ax.tick_params(labelsize=10)
ax.set_xlabel('x [pixels]', fontsize=12)
ax.set_ylabel('y [pixels]', fontsize=12)
title_obj = ax.set_title(rf'Velocity $v_x$ at $z={z}$, HII\_EFF\_FACTOR$={efflabel}$', fontsize=13)
plt.tight_layout()
plt.savefig(f"vx_z{z}_HIIeff_{effstr}.png", dpi=300)
title_obj.set_visible(False)              # Hide title
plt.savefig(f"vx_z{z}_HIIeff_{effstr}.pdf")
title_obj.set_visible(True)              #  title
plt.show()

# ========== VELOCITY Y ==========
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(vy_sl/1e5, origin='lower', cmap='magma', 
               vmin=-30, vmax=30)
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label(r'$v_y$ [$10^5$ cm s$^{-1}$]', fontsize=12)
cbar.ax.tick_params(labelsize=10)
ax.set_xlabel('x [pixels]', fontsize=12)
ax.set_ylabel('y [pixels]', fontsize=12)
title_obj = ax.set_title(rf'Velocity $v_y$ at $z={z}$, HII\_EFF\_FACTOR$={efflabel}$', fontsize=16)
plt.tight_layout()
plt.savefig(f"vy_z{z}_HIIeff_{effstr}.png", dpi=300)
title_obj.set_visible(False)              # Hide title
plt.savefig(f"vy_z{z}_HIIeff_{effstr}.pdf")
title_obj.set_visible(True)              #  title
plt.show()

# ========== VELOCITY Z ==========
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(vz_sl/1e5, origin='lower', cmap='magma', 
               vmin=-30, vmax=30)
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label(r'$v_z$ [$10^5$ cm s$^{-1}$]', fontsize=16)
cbar.ax.tick_params(labelsize=10)
ax.set_xlabel('x [pixels]', fontsize=12)
ax.set_ylabel('y [pixels]', fontsize=12)
title_obj = ax.set_title(rf'Velocity $v_z$ at $z={z}$, HII\_EFF\_FACTOR$={efflabel}$', fontsize=13)
plt.tight_layout()
plt.savefig(f"vz_z{z}_HIIeff_{effstr}.png", dpi=300)
title_obj.set_visible(False)              # Hide title
plt.savefig(f"vz_z{z}_HIIeff_{effstr}.pdf")
title_obj.set_visible(False)              # True title
plt.show()

# ========== MOMENTUM qx ==========
fig, ax = plt.subplots(figsize=(7, 6))
vmin_q, vmax_q = np.percentile(qx_sl/1e5, [2, 98])
im = ax.imshow(qx_sl/1e5, origin='lower', cmap='GnBu', 
               vmin=vmin_q, vmax=vmax_q)
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label(r'$q_x = (1+\delta) x_e v_x$ [$10^5$ cm s$^{-1}$]', fontsize=16)
cbar.ax.tick_params(labelsize=10)
ax.set_xlabel('x [pixels]', fontsize=12)
ax.set_ylabel('y [pixels]', fontsize=12)
title_obj = ax.set_title(rf'Momentum $q_x$ at $z={z}$, HII\_EFF\_FACTOR$={efflabel}$', fontsize=13)
plt.tight_layout()
plt.savefig(f"qx_z{z}_HIIeff_{effstr}.png", dpi=300)
title_obj.set_visible(False)              # Show title
plt.savefig(f"qx_z{z}_HIIeff_{effstr}.pdf")
title_obj.set_visible(False)              #  title
plt.show()

print(f"z={z}, HII_EFF_FACTOR={effval}: <x_e> = {chi.mean():.3f} | δ range = [{delta.min():.2f}, {delta.max():.2f}]")

#print(f"z={z}, HII_EFF_FACTOR={effval}:  <x_e> = {chi.mean():.3f} | δ range = [{delta.min():.2f}, {delta.max():.2f}]")
# In[ ]:


In [11]:
# ============================================================
# Cell 4: Optical Depth τ(z) and Ionisation History
# ============================================================
import os
import numpy as np
import matplotlib.pyplot as plt

# ── Build ionisation history from cached coeval runs ─────────
# Use results_ps which already has xe_mean at each redshift
zs_arr    = np.array(sorted(results_ps.keys()), dtype=float)  # ascending
xe_arr    = np.array([results_ps[z]['xe_mean'] for z in zs_arr])

# ── Comoving distances [Mpc] ──────────────────────────────────
chis = np.array([cosmo.comoving_distance(z).value for z in zs_arr])

# ── Comoving bin widths [Mpc] ─────────────────────────────────
dchi      = np.empty_like(chis)
dchi[1:]  = np.diff(chis)
dchi[0]   = dchi[1]
dchi      = np.abs(dchi)

# ── Cumulative optical depth τ(z) ────────────────────────────
# τ = σ_T * n_{e,0} * ∫ x_e(z) * (1+z)² * dχ
# where (1+z)² = a⁻² converts comoving → physical electron density
tau     = np.zeros_like(zs_arr)
running = 0.0
for i in range(len(zs_arr) - 1):
    zmid   = 0.5 * (zs_arr[i] + zs_arr[i+1])
    xe_mid = 0.5 * (xe_arr[i] + xe_arr[i+1])
    dtau   = (sigma_T * ne0 * xe_mid
              * (1.0 + zmid)**2
              * dchi[i] * MPC_CM)              # MPC_CM: Mpc → cm
    running  += dtau
    tau[i+1]  = running

print(f"τ(z_max={zs_arr.max():.1f}) = {tau[-1]:.4f}")
print(f"Planck 2018 τ_reion = 0.054 ± 0.007")

# ── ADDED: Save Reionization History & Optical Depth Data to Text File ──
txt_out = os.path.join(PLOT_DIR, "reionization_history_and_tau.txt")
header = (
    "Coeval Box Reionization History and Thomson Optical Depth Output\n"
    f"Simulation Box: {BOX_LABEL}\n"
    "Columns:\n"
    "1) redshift  : Redshift (z) in ascending order\n"
    "2) xe_mean   : Volume-averaged ionization fraction <xe>\n"
    "3) tau       : Cumulative Thomson optical depth tau(z) integrated from z_min"
)
# Stack arrays into parallel columns
data_to_save = np.column_stack((zs_arr, xe_arr, tau))
np.savetxt(txt_out, data_to_save, fmt=['%.4f', '%.6e', '%.6e'], header=header)
print(f"Text data saved → {txt_out}")

# ── Plot 1: Ionisation history x_e(z) ────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(zs_arr, xe_arr, lw=2.0, color='steelblue')
ax.set_xlabel(r'Redshift $z$')
ax.set_ylabel(r'$\langle x_e \rangle$')
ax.set_title(f'Ionisation history\n{BOX_LABEL}')
ax.axhline(0.5, color='gray', ls=':', lw=1.0, label=r'$x_e = 0.5$')
ax.axhline(1.0, color='gray', ls='--', lw=0.8, alpha=0.5)
ax.legend(frameon=False)
plt.tight_layout()
stem = os.path.join(PLOT_DIR, "ionisation_history")
plt.savefig(stem + ".pdf")
plt.savefig(stem + ".png", dpi=300)
print(f"Saved → {stem}.pdf / .png")
plt.close()

# ── Plot 2: Cumulative optical depth τ(z) ────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(zs_arr, tau, lw=2.0, color='darkorange')
ax.axhline(0.054, color='gray', ls='--', lw=1.2,
           label=r'Planck 2018: $\tau = 0.054$')
ax.axhspan(0.054 - 0.007, 0.054 + 0.007,
           color='gray', alpha=0.15, label=r'$\pm 1\sigma$')
ax.set_xlabel(r'Redshift $z$')
ax.set_ylabel(r'$\tau(z)$')
ax.set_title(f'Cumulative Thomson optical depth\n{BOX_LABEL}')
ax.legend(frameon=False)
plt.tight_layout()
stem = os.path.join(PLOT_DIR, "optical_depth")
plt.savefig(stem + ".pdf")
plt.savefig(stem + ".png", dpi=300)
print(f"Saved → {stem}.pdf / .png")
plt.close()

τ(z_max=15.0) = 0.0714
Planck 2018 τ_reion = 0.054 ± 0.007
Text data saved → 28May2026/Plots/reionization_history_and_tau.txt
Saved → 28May2026/Plots/ionisation_history.pdf / .png
Saved → 28May2026/Plots/optical_depth.pdf / .png


In [9]:
# ============================================================
# Cell 5: Transverse Momentum Power Spectrum  P_{q⊥}(k)
# ============================================================
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt

# (Assuming BOX_LEN, BOX_LABEL, PLOT_DIR, CACHE_DIR, sigma_T, ne0, const are defined above)

def qperp_power(delta, xH, vx, vy, vz, BOX_LEN=BOX_LEN, nbins=None):
    """
    P_{q⊥}(k) with TWO error estimates per bin:
      P_std    : sample std of mode powers within the shell (field scatter)
      P_cosvar : cosmic variance on the bin mean = P(k)/sqrt(N_modes)
    """
    chi = 1.0 - xH
    w   = (1.0 + delta) * chi
    qx, qy, qz = w*vx, w*vy, w*vz

    N = qx.shape[0]
    L = float(BOX_LEN)
    d = L / N
    V = L**3

    kfreq      = np.fft.fftfreq(N, d=d) * 2.0 * np.pi
    kx, ky, kz = np.meshgrid(kfreq, kfreq, kfreq, indexing='ij')
    k2         = kx**2 + ky**2 + kz**2
    k_mag      = np.sqrt(k2)
    k2_safe    = np.where(k2 == 0.0, np.inf, k2)

    Qx = np.fft.fftn(qx) * d**3
    Qy = np.fft.fftn(qy) * d**3
    Qz = np.fft.fftn(qz) * d**3

    kdotQ_k2 = (Qx*kx + Qy*ky + Qz*kz) / k2_safe
    Qx_perp  = Qx - kdotQ_k2 * kx
    Qy_perp  = Qy - kdotQ_k2 * ky
    Qz_perp  = Qz - kdotQ_k2 * kz

    Qperp2 = np.abs(Qx_perp)**2 + np.abs(Qy_perp)**2 + np.abs(Qz_perp)**2
    p_flat = (Qperp2 / V / 2.0).ravel()
    k_flat = k_mag.ravel()

    if nbins is None:
        nbins = max(2, int(np.ceil(np.cbrt(N) * 8)))

    pos_k = np.abs(kfreq[kfreq > 0.0])
    kmin  = pos_k.min() if pos_k.size > 0 else 1e-6
    kmax  = np.abs(kfreq).max() * np.sqrt(3.0)
    if kmax <= kmin:
        kmax = kmin * 10.0

    bins  = np.geomspace(kmin, kmax, nbins)
    digit = np.digitize(k_flat, bins)

    k_bins, P_bins, P_std, P_cosvar, N_modes = [], [], [], [], []
    for i in range(1, len(bins)):
        mask = digit == i
        nmodes = int(np.count_nonzero(mask))
        if nmodes == 0:
            continue
        Pmean = p_flat[mask].mean()
        k_bins.append(k_flat[mask].mean())
        P_bins.append(Pmean)
        P_std.append(p_flat[mask].std())            
        
        n_indep = max(1, nmodes // 2)
        P_cosvar.append(Pmean / np.sqrt(n_indep))   
        N_modes.append(nmodes)

    return (np.array(k_bins),
            np.array(P_bins),
            np.array(P_std),
            np.array(P_cosvar),
            np.array(N_modes))

# ── Cache helpers ─────────────────────────────────────────────
CACHE_QPERP = os.path.join(CACHE_DIR, "qperp_power.pkl")

def load_qperp_cache():
    if os.path.exists(CACHE_QPERP):
        with open(CACHE_QPERP, 'rb') as f:
            data = pickle.load(f)
        print(f"Cache loaded ← {CACHE_QPERP}  ({len(data)} redshifts)")
        return data
    return None

def save_qperp_cache(data):
    with open(CACHE_QPERP, 'wb') as f:
        pickle.dump(data, f)
    print(f"Cache saved → {CACHE_QPERP}")

# ── Redshift grid — EoR only (z ≥ 5) ─────────────────────────
ZS_EOR      = sorted([z for z in ZS_PS if z >= 5.0], reverse=True)

# !!! CHANGE 1: Set this to True once to overwrite your old cache file !!!
FORCE_RERUN = True 

results_qperp = None if FORCE_RERUN else load_qperp_cache()
if results_qperp is None:
    results_qperp = {}

missing_q = [z for z in ZS_EOR if z not in results_qperp]
if missing_q:
    print(f"Computing P_{{q⊥}} for {len(missing_q)} redshifts...\n")
    for z in missing_q:
        print(f"  z={z:.1f}...", end=' ', flush=True)
        delta, xH, vx, vy, vz = run_coeval_fields(z)
        
        # !!! CHANGE 2: Unpack all 5 variables from the function here !!!
        k_q, P_q, P_std, P_cv, N_md = qperp_power(delta, xH, vx, vy, vz, BOX_LEN)
        
        # !!! CHANGE 3: Store the new cosmic variance data in the dictionary !!!
        results_qperp[z] = {
            'k'      : k_q,
            'Pqperp' : P_q,
            'Pstd'   : P_std,
            'Pcosvar': P_cv,
            'Nmodes' : N_md,
            'xH_mean': float(xH.mean()),
        }
        print(f"done.  <xH>={xH.mean():.3f}  "
              f"P_q⊥(k~0.1) = {np.interp(0.1, k_q, P_q):.3e} cm² s⁻² Mpc³")
    save_qperp_cache(results_qperp)
    print("\nAll EoR redshifts done.")
else:
    print("Cache covers all EoR redshifts — skipping computation.")

# ── Sanity check: units ───────────────────────────────────────
z_check = 7.0
if z_check in results_qperp:
    P_at_01 = np.interp(0.1, results_qperp[z_check]['k'],
                              results_qperp[z_check]['Pqperp'])
    prefac  = (sigma_T * ne0 / const.c.cgs.value)**2   
    print(f"\nUnits check at z={z_check}, k=0.1 Mpc⁻¹:")
    print(f"  P_q⊥         = {P_at_01:.3e} cm² s⁻² Mpc³")
    print(f"  (σ_T ne0/c)² = {prefac:.3e} s² cm⁻⁴")
    print(f"  product × MPC_CM² × ds/s²a⁴ → dimensionless C_ell ✓")

# ── Plot P_{q⊥}(k) with ±1σ bands ───────────────────────────
ZS_eor_sorted = sorted(results_qperp.keys(), reverse=True)
colors_eor    = plt.cm.plasma(np.linspace(0.15, 0.85, len(ZS_eor_sorted)))

fig, ax = plt.subplots(figsize=(9, 7))
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel(r'$k\ [\mathrm{Mpc}^{-1}]$')
ax.set_ylabel(r'$P_{q_\perp}(k)\ [\mathrm{cm}^2\,\mathrm{s}^{-2}\,\mathrm{Mpc}^3]$')
ax.set_title(r'Transverse momentum power spectrum $P_{q_\perp}(k)$'
             f'\n{BOX_LABEL}')

for z, col in zip(ZS_eor_sorted, colors_eor):
    res     = results_qperp[z]
    k, P    = res['k'], res['Pqperp']
    
    # !!! CHANGE 4: Extract Cosmic Variance and Sample Standard Deviation !!!
    cv      = res['Pcosvar']
    std     = res['Pstd']
    
    ax.plot(k, P, color=col, lw=1.8)
    
    # !!! CHANGE 5: Update the plot to use Cosmic Variance as the primary shaded band !!!
    # This shows the error on the mean (narrower band)
    ax.fill_between(k, P - cv, P + cv, color=col, alpha=0.25)
    
    # (Optional) Fainter background band showing raw field scatter
    ax.fill_between(k, P - std, P + std, color=col, alpha=0.05)

norm = plt.Normalize(vmin=min(ZS_eor_sorted), vmax=max(ZS_eor_sorted))
sm   = plt.cm.ScalarMappable(cmap='plasma', norm=norm)
sm.set_array([])
cb   = fig.colorbar(sm, ax=ax, pad=0.02)
cb.set_label(r'Redshift $z$', labelpad=10)

plt.tight_layout()
stem = os.path.join(PLOT_DIR, "Pqperp")
plt.savefig(stem + ".pdf")
plt.savefig(stem + ".png", dpi=300)
print(f"\nSaved → {stem}.pdf / .png")
plt.close()

Computing P_{q⊥} for 21 redshifts...

  z=15.0... 

/home/swanith/miniconda3/envs/cmfast_tmp/lib/python3.10/site-packages/py21cmfast/inputs.py:515: UserWarning: The USE_INTERPOLATION_TABLES setting has changed in v3.1.2 to be default True. You can likely ignore this warning, but if you relied onhaving USE_INTERPOLATION_TABLES=False by *default*, please set it explicitly. To silence this warning, set it explicitly to True. Thiswarning will be removed in v4.
  warnings.warn(


done.  <xH>=0.993  P_q⊥(k~0.1) = 3.166e+13 cm² s⁻² Mpc³
  z=14.5... done.  <xH>=0.991  P_q⊥(k~0.1) = 7.374e+13 cm² s⁻² Mpc³
  z=14.0... done.  <xH>=0.986  P_q⊥(k~0.1) = 1.675e+14 cm² s⁻² Mpc³
  z=13.5... done.  <xH>=0.981  P_q⊥(k~0.1) = 3.142e+14 cm² s⁻² Mpc³
  z=13.0... done.  <xH>=0.973  P_q⊥(k~0.1) = 5.500e+14 cm² s⁻² Mpc³
  z=12.5... done.  <xH>=0.962  P_q⊥(k~0.1) = 1.292e+15 cm² s⁻² Mpc³
  z=12.0... done.  <xH>=0.947  P_q⊥(k~0.1) = 2.358e+15 cm² s⁻² Mpc³
  z=11.5... done.  <xH>=0.927  P_q⊥(k~0.1) = 4.320e+15 cm² s⁻² Mpc³
  z=11.0... done.  <xH>=0.900  P_q⊥(k~0.1) = 7.287e+15 cm² s⁻² Mpc³
  z=10.5... done.  <xH>=0.865  P_q⊥(k~0.1) = 1.135e+16 cm² s⁻² Mpc³
  z=10.0... done.  <xH>=0.820  P_q⊥(k~0.1) = 2.274e+16 cm² s⁻² Mpc³
  z=9.5... done.  <xH>=0.761  P_q⊥(k~0.1) = 4.109e+16 cm² s⁻² Mpc³
  z=9.0... done.  <xH>=0.688  P_q⊥(k~0.1) = 5.565e+16 cm² s⁻² Mpc³
  z=8.5... done.  <xH>=0.597  P_q⊥(k~0.1) = 7.571e+16 cm² s⁻² Mpc³
  z=8.0... done.  <xH>=0.489  P_q⊥(k~0.1) = 1.214e+17 cm² s⁻² M

In [70]:
# ============================================================
# Cell 5b: Δ²_{q⊥} comparison with Ma & Fry 2002
# ============================================================
# Ma & Fry 2002 (Eq. 4, Fig. 1) normalize the momentum field by H0/k:
#   q_MF = (1+δ)χv / (H0/k)   [dimensionless]
#
# Their dimensionless power spectrum is:
#   Δ²_MF(k) = k³ P_{q_MF}(k) / 2π²   [dimensionless]
#
# Ours has units:
#   Δ²_us(k) = k³ P_{q⊥}(k) / 2π²     [cm² s⁻²]
#
# Conversion:
#   Δ²_MF = Δ²_us × k² / H0²
#   where H0 in [cm/s/Mpc] and k in [Mpc⁻¹]
#   → [cm²s⁻²] × [Mpc⁻²] / [cm²s⁻²Mpc⁻²] = dimensionless ✓
#
# Ma & Fry large-k approximation (Eq. 8):
#   P_{q⊥}(k) ≈ (4/3) × P_δδ(k,z) × σ_v²(z)
#   where σ_v² = f²H²/(2π²) ∫ P_δδ(k') dk'   [cm²/s²]
# ============================================================
import camb
import pickle

# ── Get linear P_δδ from CAMB ────────────────────────────────
h = cosmo.H(0).value / 100.0

pars = camb.CAMBparams()
pars.set_cosmology(
    H0    = cosmo.H(0).value,
    ombh2 = cosmo.Ob0 * h**2,
    omch2 = (cosmo.Om0 - cosmo.Ob0) * h**2,
)
pars.InitPower.set_params(ns=0.965, As=2.1e-9)
pars.set_matter_power(redshifts=[0.0], kmax=100.0)
res_camb = camb.get_results(pars)

kh, _, pk0 = res_camb.get_matter_power_spectrum(
    minkh=1e-4, maxkh=100.0, npoints=1000)

k_lin_b5  = kh  * h           # [Mpc⁻¹]
pk_lin_b5 = pk0[0] / h**3     # [Mpc³]

print(f"CAMB: k range {k_lin_b5.min():.2e} — {k_lin_b5.max():.2e} Mpc⁻¹")

# ── Load qperp cache ──────────────────────────────────────────
CACHE_QPERP = os.path.join(CACHE_DIR, "qperp_power.pkl")
with open(CACHE_QPERP, 'rb') as f:
    results_qperp = pickle.load(f)
print(f"Loaded qperp cache: {len(results_qperp)} redshifts")

# ── Conversion factor H0 [cm/s/Mpc] ──────────────────────────
H0_cms_mpc = cosmo.H(0).value * 1e5    # [cm/s/Mpc]

# ── Box k limits ──────────────────────────────────────────────
k_box = 2.0 * np.pi / BOX_LEN          # [Mpc⁻¹]
k_nyq = np.pi / voxel                  # [Mpc⁻¹]

# ── Plot ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel(r'$k\ [h\,\mathrm{Mpc}^{-1}]$')
ax.set_ylabel(r'$\Delta^2_{q_\perp}(k) \times (k/H_0)^2$ [dimensionless]')
ax.set_title(r'$\Delta^2_{q_\perp}$ in Ma \& Fry (2002) dimensionless units'
             f'\n{BOX_LABEL}')

colors_z = plt.cm.plasma(np.linspace(0.15, 0.85, 4))
zs_check = [15.0, 10.0, 7.0, 5.0]

print(f"\n{'z':>6}  {'χ_mean':>8}  {'Δ²_dimless(k=0.1)':>20}  "
      f"{'MaFry_dimless(k=0.1)':>22}")
print("-" * 62)

for z_test, col in zip(zs_check, colors_z):
    if z_test not in results_qperp:
        print(f"  z={z_test} not in cache — skipping")
        continue
    label    = rf'$z={z_test:.0f}$'
    k_sim    = results_qperp[z_test]['k']           # [Mpc⁻¹]
    P_sim    = results_qperp[z_test]['Pqperp']       # [cm² s⁻² Mpc³]
    xH_mean  = results_qperp[z_test]['xH_mean']
    chi_mean = 1.0 - xH_mean

    # ── Our Δ² in Ma & Fry dimensionless units ────────────────
    # Δ²_dimless = k³P/(2π²) × k²/H0²
    D2_dimless = (k_sim**3 * P_sim / (2.0 * np.pi**2)
                  * k_sim**2 / H0_cms_mpc**2)

    # ── Ma & Fry large-k: P_{q⊥} = (4/3) P_δδ σ_v² ──────────
    Dz   = growth_factor(z_test)
    D0   = growth_factor(0.0)
    f_z  = growth_rate(z_test)
    H_z  = cosmo.H(z_test).value                    # [km/s/Mpc]
    pk_z = pk_lin_b5 * (Dz / D0)**2                 # [Mpc³]

    # σ_v² limited to box modes [km²/s²] → [cm²/s²]
    mask         = (k_lin_b5 >= k_box) & (k_lin_b5 <= k_nyq)
    sigma_v2_cm2 = ((f_z * H_z)**2 / (2.0 * np.pi**2)
                    * np.trapz(pk_z[mask], k_lin_b5[mask])) * 1e10

    # Interpolate P_δδ onto sim k grid
    pk_at_ksim = np.exp(np.interp(
        np.log(k_sim),
        np.log(k_lin_b5),
        np.log(pk_z + 1e-300)))

    P_MF     = (4.0/3.0) * pk_at_ksim * sigma_v2_cm2  # [cm² s⁻² Mpc³]
    D2_MF_dl = (k_sim**3 * P_MF / (2.0 * np.pi**2)
                * k_sim**2 / H0_cms_mpc**2)             # [dimensionless]

    # ── Plot vs k in h/Mpc ────────────────────────────────────
    k_h = k_sim / h

    ax.plot(k_h, D2_dimless, color=col, lw=2.0,
            label=rf'21cmFAST {label} ($\chi$={chi_mean:.2f})')
    ax.plot(k_h, D2_MF_dl,  color=col, lw=1.5, ls='--',
            label=rf'Ma\&Fry large-$k$ {label}')

    print(f"{z_test:>6.0f}  {chi_mean:>8.3f}  "
          f"{np.interp(0.1, k_sim, D2_dimless):>20.3e}  "
          f"{np.interp(0.1, k_sim, D2_MF_dl):>22.3e}")



# ── k_box and k_nyq reference lines ──────────────────────────
ax.axvline(k_box/h, color='gray', ls=':', lw=1.0, alpha=0.6)
ax.axvline(k_nyq/h, color='gray', ls=':', lw=1.0, alpha=0.6)
ax.text(k_box/h*1.05, ax.get_ylim()[0]*2,
        r'$k_\mathrm{box}$', fontsize=9, color='gray', va='bottom')
ax.text(k_nyq/h*0.5,  ax.get_ylim()[0]*2,
        r'$k_\mathrm{Nyq}$', fontsize=9, color='gray', va='bottom')

ax.legend(frameon=False, ncol=2, fontsize=9)
ax.set_xlim(5e-3/h, 1.0/h)

plt.tight_layout()
stem = os.path.join(PLOT_DIR, "Delta2_dimless_MaFry_units")
plt.savefig(stem + ".pdf")
plt.savefig(stem + ".png", dpi=300)
print(f"\nSaved → {stem}.pdf / .png")
plt.close()

CAMB: k range 6.77e-05 — 6.77e+01 Mpc⁻¹
Loaded qperp cache: 21 redshifts

     z    χ_mean     Δ²_dimless(k=0.1)    MaFry_dimless(k=0.1)
--------------------------------------------------------------
    15     0.007             9.478e-05               6.776e-02
    10     0.180             2.222e-02               9.952e-02
     7     0.767             7.700e-02               1.374e-01
     5     0.999             2.332e-02               1.831e-01

Saved → 27April2026/Plots/Delta2_dimless_MaFry_units.pdf / .png


#== Cell 5-total-q: q_total FFT and power spectrum function ===

"""
Build q=(1+δ)χv, FFT to Q(k), estimate P_{q_total}(k) (no projection).

Args:
    delta, chi, vx, vy, vz : 3D numpy arrays (shape [N,N,N]) in real space
    BOX_LEN : float, box size in Mpc (comoving)
    nbins : int or None, number of k-bins (if None a heuristic is used)
    HII_EFF_FACTOR : optional numeric label. Not used in computation,
            but returned for bookkeeping.

Returns:
    (k_bin (Mpc^-1), P_bin (cm^2 s^-2 Mpc^3),
    qtotal_slice (real-space |q| mid-slice),
    kplane_amp (|Q| mid-kz plane, fftshifted),
    kx2d, ky2d (fftshifted, for plotting axes),
    HII_EFF_FACTOR)  # final element is exactly the input HII_EFF_FACTOR (for labels/filenames)
"""

def qtotal_fft_and_power(delta, chi, vx, vy, vz, BOX_LEN, nbins=None, HII_EFF_FACTOR=None):
    """
    Robust q_total FFT + binned power estimator (no perpendicular projection).
    Returns (kvals, pvals, qtotal_slice, kplane_amp, kx2, ky2, HII_EFF_FACTOR)
    """
    
    # --- real-space q ---
    ne_fluct = (1.0 + delta) * chi
    qx = ne_fluct * vx
    qy = ne_fluct * vy
    qz = ne_fluct * vz

    # --- geometry ---
    N = qx.shape[0]
    L = float(BOX_LEN)          # Mpc (comoving)
    d = L / N                   # Mpc
    V = L**3                    # Mpc^3

    # --- k-grid (1/Mpc) ---
    kfreq = np.fft.fftfreq(N, d=d) * 2.0*np.pi  # radians per Mpc
    kx, ky, kz = np.meshgrid(kfreq, kfreq, kfreq, indexing="ij")
    k2 = kx*kx + ky*ky + kz*kz
    k = np.sqrt(k2)

    # --- FFT with continuous-FT normalization (× voxel volume) ---
    Qx = np.fft.fftn(qx) * (d**3)
    Qy = np.fft.fftn(qy) * (d**3)
    Qz = np.fft.fftn(qz) * (d**3)

    # --- power estimator: P(k) = <|Q_total|^2>/V (NO projection) ---
    Qtotal2 = (np.abs(Qx)**2 + np.abs(Qy)**2 + np.abs(Qz)**2)
    p_flat = (Qtotal2 / V).ravel()
    k_flat = k.ravel()

    # --- radial binning in k ---
    if nbins is None:
        nbins = int(np.ceil(np.cbrt(N) * 8))
    nbins = max(2, int(nbins))

    # find positive, non-zero unique kfreqs
    unique_kfreqs = np.unique(np.abs(kfreq))
    pos_kfreqs = unique_kfreqs[unique_kfreqs > 0.0]
    if pos_kfreqs.size == 0:
        kmin = 1e-6
    else:
        kmin = pos_kfreqs.min()

    kmax = np.abs(kfreq).max() * np.sqrt(3.0)
    if kmax <= kmin:
        kmax = kmin * 10.0

    bins = np.geomspace(kmin, kmax, nbins)
    digit = np.digitize(k_flat, bins)

    kvals, pvals, counts = [], [], []
    for i in range(1, len(bins)):
        mask = digit == i
        if not np.any(mask):
            continue
        kvals.append(k_flat[mask].mean())
        pvals.append(p_flat[mask].mean())
        counts.append(int(mask.sum()))

    # --- optional visuals: real-space |q_total| mid-slice, and k-plane amplitude (fftshifted) ---
    qx_r = np.fft.ifftn(Qx) / (d**3)
    qy_r = np.fft.ifftn(Qy) / (d**3)
    qz_r = np.fft.ifftn(Qz) / (d**3)
    qtotal_mag = np.sqrt(np.abs(qx_r)**2 + np.abs(qy_r)**2 + np.abs(qz_r)**2).real

    mid = N//2
    qtotal_slice = qtotal_mag[mid, :, :]

    plane_Q = (np.abs(Qx)**2 + np.abs(Qy)**2 + np.abs(Qz)**2)**0.5
    kplane_amp = np.fft.fftshift(plane_Q[:, :, mid])

    kx2 = np.fft.fftshift(kx[:, :, mid])
    ky2 = np.fft.fftshift(ky[:, :, mid])

    return (np.array(kvals), np.array(pvals),
            qtotal_slice, kplane_amp, kx2, ky2, HII_EFF_FACTOR)

#=== Cell 5b: Diagnostic loop: tune z_show and HII_EFF_values here ===
z_show = 12.0
HII_EFF_values = [50.0]#[1.0, 10.0, 50.0]   # low to high efficiency

import math

for effval in HII_EFF_values:
    print(f"\nRunning diagnostics at z={z_show}, HII_EFF_FACTOR={effval} ...")

    # 1) generate fields for this (z, HII_EFF_FACTOR)
    delta, chi, vx, vy, vz = run_coeval_fields(z_show, astro_overrides={"HII_EFF_FACTOR": effval})
    qx, qy, qz = build_momentum(delta, chi, vx, vy, vz, z_show)

    # 2) compute FFT / power / k-plane using your function
    kvals, Pbins, qperp_slice, kplane_amp, kplane_amp_perp, kx2, ky2, eff_used = qperp_fft_and_power(
        delta, chi, vx, vy, vz, BOX_LEN, nbins=None, HII_EFF_FACTOR=effval, los_direction=[0,0,1]
    )

    # ========== STATISTICS FOR Q_perp (k-space amplitude) ==========
    print("\n" + "="*60)
    print("STATISTICS: |Q_perp(k)| in k-space")
    print("="*60)
    print(f"Min:    {kplane_amp_perp.min():.3e} cm/s Mpc³")
    print(f"Max:    {kplane_amp_perp.max():.3e} cm/s Mpc³")
    print(f"Mean:   {kplane_amp_perp.mean():.3e} cm/s Mpc³")
    print(f"Median: {np.median(kplane_amp_perp):.3e} cm/s Mpc³")
    print(f"Std:    {kplane_amp_perp.std():.3e} cm/s Mpc³")
    
    # ========== STATISTICS FOR P_q_perp ==========
    print("\n" + "="*60)
    print("STATISTICS: P_q_perp(k) power spectrum")
    print("="*60)
    print(f"Min:    {Pbins.min():.3e} cm² s⁻² Mpc³")
    print(f"Max:    {Pbins.max():.3e} cm² s⁻² Mpc³")
    print(f"Mean:   {Pbins.mean():.3e} cm² s⁻² Mpc³")
    print(f"Median: {np.median(Pbins):.3e} cm² s⁻² Mpc³")
    print(f"Std:    {Pbins.std():.3e} cm² s⁻² Mpc³")
    print(f"k range: [{kvals.min():.3e}, {kvals.max():.3e}] Mpc⁻¹")
    print(f"Number of k-bins: {len(kvals)}")
    print("="*60 + "\n")

    # 3) rebuild mode-counts / centers to match estimator (same logic as function)
    N = delta.shape[0]
    L = BOX_LEN
    d = L / N
    kfreq = np.fft.fftfreq(N, d=d) * 2.0*np.pi
    

        # robust non-zero kfreqs
    # Get all unique k-frequency values and take absolute value
    nonzero_kfreqs = np.abs(np.unique(kfreq))
    # Filter to keep only positive frequencies; fallback to 1e-6 if none exist
    nonzero_kfreqs = nonzero_kfreqs[nonzero_kfreqs > 0] if nonzero_kfreqs.size>1 else np.array([1e-6])
    # Set minimum k for binning (smallest non-zero mode in the box)
    kmin = nonzero_kfreqs.min()
    # Set maximum k for binning: Nyquist frequency × sqrt(3) for 3D diagonal modes
    # Nyquist freq k_Nyq = π/Δx is the highest frequency resolvable on the grid
    # In 3D, the corner mode has |k| = sqrt(k_x² + k_y² + k_z²) = sqrt(3) × k_Nyq
    # when k_x = k_y = k_z = k_Nyq, so we multiply by sqrt(3) to include all modes
    kmax = kfreq.max() * np.sqrt(3.0)
    # Number of logarithmic bins: heuristic ~8 × N^(1/3)
    nb = int(np.ceil(np.cbrt(N) * 8))
    # Create logarithmically-spaced bin edges from kmin to kmax
    bins = np.geomspace(kmin, kmax, nb)
    # Create 3D meshgrid of k-vectors in Fourier space
    kxg, kyg, kzg = np.meshgrid(kfreq, kfreq, kfreq, indexing="ij")
    # Compute magnitude |k| = sqrt(kx² + ky² + kz²) for each grid point, flatten to 1D
    k3d = np.sqrt(kxg*kxg + kyg*kyg + kzg*kzg).ravel()
    # Assign each k-mode to a radial bin (returns bin index for each k-mode)
    digit = np.digitize(k3d, bins)

    centers = []
    counts = []
    for i in range(1, len(bins)):
        mask = digit == i
        if not np.any(mask):
            continue
        centers.append(k3d[mask].mean())
        counts.append(mask.sum())
    centers = np.array(centers)
    counts = np.array(counts)

    # 4) pretty HII_EFF_FACTOR label
    if eff_used is None:
        efflabel = "HII_EFF_FACTOR=?"
        fnamefrag = "HIIeff_unknown"
        exp = 0
    else:
        exp = int(math.floor(math.log10(eff_used))) if eff_used > 0 else 0
        coeff = eff_used / (10**exp) if eff_used > 0 else eff_used
        if abs(coeff - 1.0) < 1e-3:
            efflabel = fr"$10^{{{exp}}}$"
            fnamefrag = f"1e{exp}"
        else:
            efflabel = fr"${coeff:.1f}\times 10^{{{exp}}}$"
            fnamefrag = f"{coeff:.1f}e{exp}".replace('.', 'p')

    # -----------------------
# (B) k-space mid-kz plane (|Q_perp|) WITH UNITS
# -----------------------
fig, ax = plt.subplots(figsize=(7, 6.5))
im = ax.imshow(kplane_amp_perp, origin="lower", cmap="magma", norm=LogNorm(),
               extent=[kx2.min(), kx2.max(), ky2.min(), ky2.max()], aspect="equal")
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label(r"$|Q_\perp(k_x,k_y,k_z{=}0)|$ [cm s$^{-1}$ Mpc$^3$]", fontsize=16)
cbar.ax.tick_params(labelsize=14)
ax.set_xlabel(r"$k_x$ [Mpc$^{-1}$]", fontsize=16)
ax.set_ylabel(r"$k_y$ [Mpc$^{-1}$]", fontsize=16)
ax.tick_params(labelsize=14)
title_obj = ax.set_title(
    f"k-plane amplitude of $Q_\\perp$ (z={z_show}, HII_EFF_FACTOR={efflabel})\n"
    f"Box: {L} Mpc, N={N} ($\\Delta x$={d:.2f} Mpc)", 
    fontsize=16
)
outname_kplane = f"q_perp_mid_kz_z{z_show}_HIIeff_{fnamefrag}.png"
plt.tight_layout()
plt.savefig(outname_kplane, dpi=300)  # Save PNG with title
title_obj.set_visible(False)  # Hide title
plt.savefig(outname_kplane.replace('.png', '.pdf'))  # Save PDF without title
plt.show()
plt.close()

# -----------------------
# (C) radial shell counts in k
# -----------------------
fig, ax = plt.subplots(figsize=(7, 6))
ax.loglog(centers, counts, marker="o", ls="none", ms=6)
ax.set_xlabel(r"$k$ [Mpc$^{-1}$]", fontsize=16)
ax.set_ylabel("Modes per shell", fontsize=16)
ax.tick_params(labelsize=14)
title_obj = ax.set_title(
    f"Radial shell counts in k (z={z_show}, HII_EFF_FACTOR={efflabel})\n"
    f"Box: {L} Mpc, N={N}", 
    fontsize=16
)
outname_counts = f"number_of_modes_in_bins_z{z_show}_HIIeff_{fnamefrag}.png"
plt.tight_layout()
plt.savefig(outname_counts, dpi=300)  # Save PNG with title
title_obj.set_visible(False)  # Hide title
plt.savefig(outname_counts.replace('.png', '.pdf'))  # Save PDF without title
plt.show()
plt.close()

# -----------------------
# (D) binned power spectrum
# -----------------------
fig, ax = plt.subplots(figsize=(7, 6))
ax.loglog(kvals, Pbins, lw=2.0, marker='o', ms=5)
ax.set_xlabel(r"$k$ [Mpc$^{-1}$]", fontsize=16)
ax.set_ylabel(r"$P_{q_\perp}(k)$ [cm$^2$ s$^{-2}$ Mpc$^3$]", fontsize=16)
ax.tick_params(labelsize=14)
title_obj = ax.set_title(
    f"$P_{{q_\\perp}}(k)$ at z={z_show}, HII_EFF_FACTOR={efflabel}\n"
    f"Box: {L} Mpc, N={N}", 
    fontsize=16
)
outname_pk = f"P_q_perp_z{z_show}_HIIeff_{fnamefrag}.png"
plt.tight_layout()
plt.savefig(outname_pk, dpi=300)  # Save PNG with title
title_obj.set_visible(False)  # Hide title
plt.savefig(outname_pk.replace('.png', '.pdf'))  # Save PDF without title
plt.show()
plt.close()

print(f"Saved diagnostics: {outname_kplane}, {outname_counts}, {outname_pk}")

# ============================================================
# Cell 6: P_{q⊥}(k) across HII_EFF_FACTOR values
# ============================================================
# Loops over astrophysical parameter HII_EFF_FACTOR, computing
# P_{q⊥}(k) at each redshift. Two plotting modes:
#   'by_HIIeff' : one figure per HII_EFF_FACTOR, curves = redshifts
#   'by_z'      : one figure per redshift, curves = HII_EFF_FACTOR
#
# Shaded bands show ±1σ from radial k-bin scatter.
# ============================================================
import pickle

# ── Configuration ─────────────────────────────────────────────
HII_EFF_VALUES = [10.0, 50.0, 100.0]          # ζ values to compare
ZS_EOR         = sorted([z for z in ZS_PS
                          if z >= 5.0], reverse=True)
MODE           = 'by_HIIeff'                   # 'by_HIIeff' or 'by_z'
FORCE_RERUN    = False

# ── Cache helpers ─────────────────────────────────────────────
CACHE_CELL6 = os.path.join(CACHE_DIR, "qperp_HIIeff.pkl")

def load_cell6_cache():
    if os.path.exists(CACHE_CELL6):
        with open(CACHE_CELL6, 'rb') as f:
            data = pickle.load(f)
        print(f"Cache loaded ← {CACHE_CELL6}")
        return data
    return {}

def save_cell6_cache(data):
    with open(CACHE_CELL6, 'wb') as f:
        pickle.dump(data, f)
    print(f"Cache saved → {CACHE_CELL6}")

# ── Load or compute ───────────────────────────────────────────
# results keyed by (z, HII_EFF_FACTOR)
results_cell6 = {} if FORCE_RERUN else load_cell6_cache()

for effval in HII_EFF_VALUES:
    for z in ZS_EOR:
        key = (z, effval)
        if key in results_cell6:
            continue
        print(f"  z={z:.1f}  HII_EFF={effval:.0f}...", end=' ', flush=True)
        delta, xH, vx, vy, vz = run_coeval_fields(
            z, astro_overrides={"HII_EFF_FACTOR": effval})
        k_q, P_q, P_std = qperp_power(delta, xH, vx, vy, vz, BOX_LEN)
        results_cell6[key] = {
            'k'      : k_q,
            'Pqperp' : P_q,
            'Pstd'   : P_std,
            'xH_mean': float(xH.mean()),
        }
        print(f"<xH>={xH.mean():.3f}  "
              f"P_q⊥(k~0.1)={np.interp(0.1, k_q, P_q):.3e}")

save_cell6_cache(results_cell6)
print("\nAll done.")

# ── Colour maps ───────────────────────────────────────────────
cmap_z   = plt.cm.plasma
cmap_eff = plt.cm.viridis

# ── Mode A: one figure per HII_EFF_FACTOR ────────────────────
if MODE == 'by_HIIeff':
    for effval in HII_EFF_VALUES:
        colors_z = cmap_z(np.linspace(0.15, 0.85, len(ZS_EOR)))

        fig, ax = plt.subplots(figsize=(9, 7))
        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_xlabel(r'$k\ [\mathrm{Mpc}^{-1}]$')
        ax.set_ylabel(r'$P_{q_\perp}(k)\ [\mathrm{cm}^2\,\mathrm{s}^{-2}\,\mathrm{Mpc}^3]$')
        ax.set_title(r'$P_{q_\perp}(k)$ — '
                     rf'$\zeta={effval:.0f}$'
                     f'\n{BOX_LABEL}')

        for z, col in zip(ZS_EOR, colors_z):
            key = (z, effval)
            if key not in results_cell6:
                continue
            res = results_cell6[key]
            k, P, S = res['k'], res['Pqperp'], res['Pstd']

            ax.plot(k, P, color=col, lw=1.8)
            ax.fill_between(k, P - S, P + S,
                            color=col, alpha=0.15)   # ±1σ band

        norm = plt.Normalize(vmin=min(ZS_EOR), vmax=max(ZS_EOR))
        sm   = plt.cm.ScalarMappable(cmap='plasma', norm=norm)
        sm.set_array([])
        cb   = fig.colorbar(sm, ax=ax, pad=0.02)
        cb.set_label(r'Redshift $z$', labelpad=10)

        plt.tight_layout()
        stem = os.path.join(PLOT_DIR,
               f"Pqperp_HIIeff{effval:.0f}")
        plt.savefig(stem + ".pdf")
        plt.savefig(stem + ".png", dpi=300)
        print(f"Saved → {stem}.pdf / .png")
        plt.close()

# ── Mode B: one figure per redshift ──────────────────────────
elif MODE == 'by_z':
    colors_eff = cmap_eff(np.linspace(0.15, 0.85, len(HII_EFF_VALUES)))

    for z in ZS_EOR:
        fig, ax = plt.subplots(figsize=(9, 7))
        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_xlabel(r'$k\ [\mathrm{Mpc}^{-1}]$')
        ax.set_ylabel(r'$P_{q_\perp}(k)\ [\mathrm{cm}^2\,\mathrm{s}^{-2}\,\mathrm{Mpc}^3]$')
        ax.set_title(r'$P_{q_\perp}(k)$ — '
                     rf'$z={z:.1f}$'
                     f'\n{BOX_LABEL}')

        for effval, col in zip(HII_EFF_VALUES, colors_eff):
            key = (z, effval)
            if key not in results_cell6:
                continue
            res = results_cell6[key]
            k, P, S = res['k'], res['Pqperp'], res['Pstd']

            ax.plot(k, P, color=col, lw=1.8,
                    label=rf'$\zeta={effval:.0f}$')
            ax.fill_between(k, P - S, P + S,
                            color=col, alpha=0.15)

        ax.legend(frameon=False, fontsize=12)
        plt.tight_layout()
        stem = os.path.join(PLOT_DIR,
               f"Pqperp_z{z:.1f}")
        plt.savefig(stem + ".pdf")
        plt.savefig(stem + ".png", dpi=300)
        print(f"Saved → {stem}.pdf / .png")
        plt.close()

else:
    raise ValueError("MODE must be 'by_HIIeff' or 'by_z'")

In [10]:
# ============================================================
# Cell 7: kSZ Angular Power Spectrum  C_ell / D_ell
# ============================================================
# Implements Cain+2024 Eq. (3) / Park+2013 Appendix Eq. (A16):
#
#   C_ell = (σ_T ne0 / c)² ∫ e^{-2τ} / (s² a⁴)  P_{q⊥}(k=ℓ/s)  ds
#
# Error propagation (Independent slices in quadrature):
#   σ²_Cell_std    = pref² × Σ_i (w_i × σ_P_std_i)² × MPC_CM⁴
#   σ²_Cell_cosvar = pref² × Σ_i (w_i × σ_P_cv_i)² × MPC_CM⁴
# ============================================================
import pickle
import os
import numpy as np
import matplotlib.pyplot as plt

# ── Physical constants ────────────────────────────────────────
c_cms   = const.c.cgs.value           # speed of light  [cm s⁻¹]
T_CMB_K = 2.7255                       # CMB temperature [K]

# ── Prefactor (σ_T ne0 / c)² ─────────────────────────────────
pref = (sigma_T * ne0 / c_cms)**2     # [s² cm⁻⁴]

# ── Full multipole grid (will be restricted below) ────────────
ells_full = np.unique(
    np.round(np.logspace(2.0, 4.5, 80)).astype(int)
).astype(int)

# ── Load qperp cache from Cell 5 (Contains Pcosvar!) ──────────
CACHE_QPERP = os.path.join(CACHE_DIR, "qperp_power.pkl")
with open(CACHE_QPERP, 'rb') as f:
    results_qperp = pickle.load(f)
print(f"Loaded P_{{q⊥}} cache: {len(results_qperp)} redshifts")

# ── Redshift grid — EoR only, ascending for τ integration ─────
ZS_asc = sorted([z for z in results_qperp.keys() if z >= 5.0])
print(f"Integrating over {len(ZS_asc)} redshift slices: "
      f"z={ZS_asc[0]:.1f} → z={ZS_asc[-1]:.1f}")

# ── Comoving distances ────────────────────────────────────────
chi_mpc  = np.array([cosmo.comoving_distance(z).value for z in ZS_asc]) # [Mpc]
dchi_mpc = np.abs(np.gradient(chi_mpc))                                 # [Mpc]
dchi_cm  = dchi_mpc * MPC_CM                                            # [cm] — τ only

# ── Valid ell range ───────────────────────────────────────────
k_max_sim     = min(results_qperp[z]['k'].max() for z in ZS_asc)
k_min_sim     = max(results_qperp[z]['k'].min() for z in ZS_asc)
s_min         = chi_mpc.min()
s_max         = chi_mpc.max()

ell_max_valid = int(k_max_sim * s_min)
ell_min_valid = int(k_min_sim * s_max)

ells = ells_full[(ells_full >= ell_min_valid) & (ells_full <= ell_max_valid)]
print(f"\nk range (sim)  : {k_min_sim:.4f} — {k_max_sim:.4f} Mpc⁻¹")
print(f"s range        : {s_min:.0f} — {s_max:.0f} Mpc")
print(f"Valid ell range: {ell_min_valid} — {ell_max_valid}")
print(f"Using {len(ells)} ell bins: {ells.min()} — {ells.max()}")

# ── Pull fields from cache (including Cosmic Variance) ────────
k_list, P_list, S_list, CV_list, xe_list = [], [], [], [], []
for z in ZS_asc:
    k_list.append(results_qperp[z]['k'])
    P_list.append(results_qperp[z]['Pqperp'])
    S_list.append(results_qperp[z]['Pstd'])          # Field scatter std
    CV_list.append(results_qperp[z]['Pcosvar'])      # Cosmic variance on the mean
    xe_list.append(1.0 - results_qperp[z]['xH_mean'])

ZS_asc  = np.array(ZS_asc,  dtype=float)
xe_arr  = np.array(xe_list, dtype=float)

# ── Optical depth τ(z) ───────────────────────────────────────
tau = np.zeros_like(ZS_asc, dtype=float)
for i in range(len(ZS_asc) - 1):
    zmid   = 0.5 * (ZS_asc[i]  + ZS_asc[i+1])
    xe_mid = 0.5 * (xe_arr[i]  + xe_arr[i+1])
    tau[i+1] = tau[i] + (sigma_T * ne0 * xe_mid * (1.0 + zmid)**2 * dchi_cm[i])

print(f"\nτ(z_max={ZS_asc[-1]:.1f}) = {tau[-1]:.4f}  (Planck 2018: 0.054 ± 0.007)")

# ── Log-log interpolation helper ─────────────────────────────
def interp_loglog(xq, xp, fp):
    xp = np.asarray(xp);  fp = np.asarray(fp)
    m  = (xp > 0) & (fp > 0)
    lx = np.log(xp[m]);   lf = np.log(fp[m])
    lq = np.log(np.clip(xq, xp[m].min(), xp[m].max()))
    return np.exp(np.interp(lq, lx, lf))

# ── C_ell Line-of-Sight Integral with Two Error Propagations ──
print("\nComputing C_ell integral with dual error propagation...")
C_ell          = np.zeros(len(ells), dtype=float)
var_C_ell_std  = np.zeros(len(ells), dtype=float)   # Field scatter variance
var_C_ell_cv   = np.zeros(len(ells), dtype=float)   # Cosmic variance variance

for i in range(len(ZS_asc)):
    s_mpc = chi_mpc[i]
    if s_mpc <= 0.0:
        continue
    a_i  = 1.0 / (1.0 + ZS_asc[i])
    vis2 = np.exp(-2.0 * tau[i])
    w    = vis2 / (s_mpc**2 * a_i**4) * dchi_mpc[i]     # [Mpc⁻¹]
    k_ell = ells / s_mpc                                # [Mpc⁻¹]

    # Central value
    P_now = interp_loglog(k_ell, k_list[i], P_list[i])
    C_ell += pref * w * P_now * MPC_CM**2

    # Standard Deviation (Field Scatter) Propagation
    S_now          = interp_loglog(k_ell, k_list[i], S_list[i])
    var_C_ell_std += (pref * w * S_now * MPC_CM**2)**2

    # Cosmic Variance Propagation
    CV_now         = interp_loglog(k_ell, k_list[i], CV_list[i])
    var_C_ell_cv  += (pref * w * CV_now * MPC_CM**2)**2

sigma_C_ell_std = np.sqrt(var_C_ell_std)
sigma_C_ell_cv  = np.sqrt(var_C_ell_cv)

# ── D_ell and Errors [μK²] ───────────────────────────────────
prefactor_D = ells * (ells + 1.0) / (2.0 * np.pi) * T_CMB_K**2 * 1e12

D_ell              = prefactor_D * C_ell
sigma_D_ell_std    = prefactor_D * sigma_C_ell_std
sigma_D_ell_cosvar = prefactor_D * sigma_C_ell_cv

D3000   = float(np.interp(3000.0, ells, D_ell))
cv3000  = float(np.interp(3000.0, ells, sigma_D_ell_cosvar))
std3000 = float(np.interp(3000.0, ells, sigma_D_ell_std))
print(f"D_3000 = {D3000:.4g} (± {cv3000:.4g} Cosmic Var) (± {std3000:.4g} Field Std) μK²")

# ── Save updated pickle cache ─────────────────────────────────
CACHE_CELL7 = os.path.join(CACHE_DIR, "ksz_Dell.pkl")
results_cell7 = {
    'ells'            : ells.copy(),
    'D_ell'           : D_ell.copy(),
    'sigma_D_std'     : sigma_D_ell_std.copy(),
    'sigma_D_cosvar'  : sigma_D_ell_cosvar.copy(),
    'C_ell'           : C_ell.copy(),
    'sigma_C_std'     : sigma_C_ell_std.copy(),
    'sigma_C_cosvar'  : sigma_C_ell_cv.copy(),
    'tau'             : (ZS_asc.copy(), tau.copy()),
    'xe'              : (ZS_asc.copy(), xe_arr.copy()),
}
with open(CACHE_CELL7, 'wb') as f:
    pickle.dump(results_cell7, f)
print(f"Cache saved → {CACHE_CELL7}")

# ── Save text file of D_ell and errors ───────────────────────
TXT_OUT = os.path.join(PLOT_DIR, "ksz_dell_spectrum.txt")
header = (
    "kSZ Angular Power Spectrum Output\n"
    f"Simulation Box: {BOX_LABEL}\n"
    "Columns:\n"
    "1) ell              : Multipole moment\n"
    "2) D_ell            : kSZ power spectrum [muK^2]\n"
    "3) sigma_D_cosvar   : 1-sigma cosmic variance uncertainty on the mean [muK^2]\n"
    "4) sigma_D_std      : 1-sigma field sample standard deviation [muK^2]"
)
data_to_save = np.column_stack((ells, D_ell, sigma_D_ell_cosvar, sigma_D_ell_std))
np.savetxt(TXT_OUT, data_to_save, fmt=['%d', '%.6e', '%.6e', '%.6e'], header=header)
print(f"Text data saved → {TXT_OUT}")

# ── Plot 1: D_ell kSZ power spectrum with BOTH error bands ───
fig, ax = plt.subplots(figsize=(9, 7))
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel(r'Multipole $\ell$')
ax.set_ylabel(r'$D_\ell\ [\mu\mathrm{K}^2]$')
ax.set_title(r'Patchy kSZ Power Spectrum  ($z \geq 5$, default $\zeta$)'
             f'\n{BOX_LABEL}')

ax.plot(ells, D_ell, color='steelblue', lw=2.5, label='This work')

# Primary band: Cosmic Variance (Error on the binned spectrum mean)
ax.fill_between(ells,
                np.maximum(D_ell - sigma_D_ell_cosvar, 1e-6),
                D_ell + sigma_D_ell_cosvar,
                color='steelblue', alpha=0.30,
                label=r'$\pm 1\sigma$ Cosmic Variance')

# Secondary band: Field Scatter (Raw mode standard deviation)
ax.fill_between(ells,
                np.maximum(D_ell - sigma_D_ell_std, 1e-6),
                D_ell + sigma_D_ell_std,
                color='steelblue', alpha=0.07,
                label=r'$\pm 1\sigma$ Mode Scatter ($\sigma_{\mathrm{std}}$)')

# Reichardt+2021 measurement
ax.errorbar(3000, 1.1, yerr=[[0.7], [1.0]],
            fmt='s', ms=8, capsize=5, capthick=2,
            color='red', zorder=10, label='Reichardt et al. (2021)')

ax.set_xlim(ells.min() * 0.8, ells.max() * 1.2)
ax.legend(frameon=False, fontsize=11, loc='lower left')
plt.tight_layout()
stem = os.path.join(PLOT_DIR, "Dell_ksz")
plt.savefig(stem + ".pdf")
plt.savefig(stem + ".png", dpi=300)
plt.close()

# ── Plots 2 & 3: Ionisation history and Optical Depth ────────
# (Kept identical to original outputs for pipeline completeness)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(ZS_asc, xe_arr, color='steelblue', lw=2.0)
ax.axhline(0.5, color='gray', ls=':', lw=1.0, label=r'$x_e = 0.5$')
ax.set_xlabel(r'Redshift $z$'); ax.set_ylabel(r'$\langle x_e \rangle$')
ax.set_title(f'Ionisation History (default $\\zeta$)\n{BOX_LABEL}')
ax.legend(frameon=False); plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "xe_history_ksz.png"), dpi=300); plt.close()

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(ZS_asc, tau, color='steelblue', lw=2.0, label='This work')
ax.axhline(0.054, color='gray', ls='--', lw=1.2, label=r'Planck 2018: $\tau = 0.054$')
ax.axhspan(0.047, 0.061, color='gray', alpha=0.15, label=r'$\pm 1\sigma$')
ax.set_xlabel(r'Redshift $z$'); ax.set_ylabel(r'$\tau(z)$')
ax.set_title(f'Cumulative Thomson Optical Depth\n{BOX_LABEL}')
ax.legend(frameon=False); plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "tau_ksz.png"), dpi=300); plt.close()

print("All plots updated and saved.")

Loaded P_{q⊥} cache: 21 redshifts
Integrating over 21 redshift slices: z=5.0 → z=15.0

k range (sim)  : 0.0079 — 0.7977 Mpc⁻¹
s range        : 7946 — 10453 Mpc
Valid ell range: 82 — 6338
Using 57 ell bins: 100 — 5918

τ(z_max=15.0) = 0.0342  (Planck 2018: 0.054 ± 0.007)

Computing C_ell integral with dual error propagation...
D_3000 = 0.6952 (± 0.0009112 Cosmic Var) (± 0.1507 Field Std) μK²
Cache saved → 28May2026/Cache/ksz_Dell.pkl
Text data saved → 28May2026/Plots/ksz_dell_spectrum.txt
All plots updated and saved.


# ==========cell 7.5 -  IONIZATION HISTORY WITH OBSERVATIONAL DATA ==========
fig, ax = plt.subplots(figsize=(10, 7))

# Plot your simulation results
for effval in HII_EFF_values:
    _, frag, _ = efflabel_and_frag(effval)
    zs_plot, xe_plot = xe_by_m[frag]
    order_plot = np.argsort(zs_plot)
    ax.plot(zs_plot[order_plot], xe_plot[order_plot], 
            marker='o', ms=8, lw=3.0, 
            label=f"This work (HII_EFF={efflabel_and_frag(effval)[0]})",
            color='#2E86AB', zorder=10)

# --- Color scheme by measurement technique ---
# Black/Gray = QSO damping wing measurements
# Blue = LAE Luminosity Functions
# Green = LAE Clustering (ACF)
# Red = Lyα forest / GP troughs

# ============ QSO DAMPING WING MEASUREMENTS (Black/Gray) ============

# Mason et al. 2018 - QSO damping wing
x_measurement = 1 - 0.64
x_err_lower = 0.36 - (1 - (0.64 + 0.13))
x_err_upper = (1 - (0.64 - 0.21)) - 0.36
z_err = 0.5

ax.errorbar(7, x_measurement,
            yerr=[[x_err_lower], [x_err_upper]],
            xerr=z_err,
            fmt='o', mfc='white', mec='black', ecolor='black',
            capsize=5, markersize=8,
            label='QSO damping wing', zorder=5)

# Davies et al. 2018 - QSO damping wing
z_mid_8 = (7.4 + 8.0) / 2
z_err_8 = (8.0 - 7.4) / 2
xHI_8 = 0.62
xHII_8 = 1 - xHI_8
xHI_err_lower_8 = 0.36
xHI_err_upper_8 = 0.15
xHII_err_lower_8 = xHI_err_upper_8
xHII_err_upper_8 = xHI_err_lower_8

ax.errorbar(z_mid_8, xHII_8,
            yerr=[[xHII_err_lower_8], [xHII_err_upper_8]],
            xerr=z_err_8,
            fmt='o', mfc='white', mec='black', ecolor='black',
            capsize=5, markersize=8,
            label='_nolegend_', zorder=5)

# Greig et al. 2017 - QSO (dark fraction method)
z_mid_7 = (6.6 + 7.3) / 2
z_err_7 = (7.3 - 6.6) / 2
xHI_7 = 0.79
xHII_7 = 1 - xHI_7

ax.errorbar(z_mid_7, xHII_7, 
            xerr=z_err_7,
            yerr=0.21,
            lolims=True,
            fmt='o', mfc='white', mec='black', ecolor='black',
            capsize=5, markersize=8,
            label='_nolegend_', zorder=5)

# Morishita et al. 2023 - QSO
z_morishita = 7.88
xHI_lower = 0.45
xHII_upper = 1 - xHI_lower

ax.errorbar(z_morishita, xHII_upper,
            yerr=0.05,
            uplims=True,
            fmt='o', mfc='white', mec='black', ecolor='black',
            capsize=5, markersize=8,
            label='_nolegend_', zorder=5)

# ============ CMB OPTICAL DEPTH (Gray) ============

# Planck 2016 - CMB τ constraint
z_range = (9 + 13) / 2
z_err_range = (13 - 9) / 2
xHI_9_13 = 0.93
xHII_9_13 = 1 - xHI_9_13
xHI_err_lower_9_13 = 0.07
xHI_err_upper_9_13 = 0.04
xHII_err_lower_9_13 = xHI_err_upper_9_13
xHII_err_upper_9_13 = xHI_err_lower_9_13

ax.errorbar(z_range, xHII_9_13,
            yerr=[[xHII_err_lower_9_13], [xHII_err_upper_9_13]],
            xerr=z_err_range,
            fmt='s', mfc='lightgray', mec='gray', ecolor='gray',
            capsize=5, markersize=8,
            label='CMB optical depth', zorder=5)

# ============ LAE LUMINOSITY FUNCTIONS (Blue shades) ============

# Umeda et al. 2024 - LAE LF
ax.errorbar(5.7, 1 - 0.05,
            yerr=0.05,
            lolims=True,
            fmt='^', mfc='lightblue', mec='blue', ecolor='blue',
            capsize=5, markersize=8,
            label='LAE luminosity function', zorder=5)

ax.errorbar(6.6, 1 - 0.15,  
            yerr=[[0.10], [0.08]],
            fmt='^', mfc='lightblue', mec='blue', ecolor='blue',
            capsize=5, markersize=8,
            label='_nolegend_', zorder=5)

ax.errorbar(7.0, 1 - 0.18,  
            yerr=[[0.14], [0.12]],
            fmt='^', mfc='lightblue', mec='blue', ecolor='blue',
            capsize=5, markersize=8,
            label='_nolegend_', zorder=5)

ax.errorbar(7.3, 1 - 0.75,  
            yerr=[[0.09], [0.13]],
            fmt='^', mfc='lightblue', mec='blue', ecolor='blue',
            capsize=5, markersize=8,
            label='_nolegend_', zorder=5)

# Ning et al. 2022 - LAE LF
ax.errorbar(6.6, 1 - 0.3, 
            yerr=[[0.1], [0.1]],
            fmt='^', mfc='lightblue', mec='blue', ecolor='blue',
            capsize=5, markersize=8,
            label='_nolegend_', zorder=5)

# Morales et al. 2021 - LAE LF
ax.errorbar(6.6, 1 - 0.08, 
            yerr=[[0.08], [0.05]],
            fmt='^', mfc='lightblue', mec='blue', ecolor='blue',
            capsize=5, markersize=8,
            label='_nolegend_', zorder=5)

ax.errorbar(7.0, 1 - 0.28, 
            yerr=0.05, 
            fmt='^', mfc='lightblue', mec='blue', ecolor='blue',
            capsize=5, markersize=8,
            label='_nolegend_', zorder=5)

ax.errorbar(7.3, 1 - 0.83, 
            yerr=[[0.06], [0.07]],
            fmt='^', mfc='lightblue', mec='blue', ecolor='blue',
            capsize=5, markersize=8,
            label='_nolegend_', zorder=5)

# ============ LAE CLUSTERING / ACF (Green) ============

# Umeda et al. 2024 - ACF
ax.errorbar(5.7, 1 - 0.06,  
            yerr=[[0.12], [0.03]],
            fmt='D', mfc='lightgreen', mec='green', ecolor='green',
            capsize=5, markersize=7,
            label='LAE clustering (ACF)', zorder=5)

ax.errorbar(6.6, 1 - 0.21,  
            yerr=[[0.19], [0.14]],
            fmt='D', mfc='lightgreen', mec='green', ecolor='green',
            capsize=5, markersize=7,
            label='_nolegend_', zorder=5)

# ============ Lyα FOREST / GP TROUGHS (Red) ============

# Whitler et al. 2020 - GP troughs
ax.errorbar(7.3, 1 - 0.28, 
            yerr=0.1, 
            uplims=True,
            fmt='v', mfc='lightcoral', mec='red', ecolor='red',
            capsize=5, markersize=8,
            label=r'Ly$\alpha$ forest / GP troughs', zorder=5)

# Bruton et al. 2023 - Lyα damping wing
z_bruton = 10.6
xHI_upper = 0.88
xHII_lower = 1 - xHI_upper

ax.errorbar(z_bruton, xHII_lower,
            yerr=0.05,
            lolims=True,
            fmt='v', mfc='lightcoral', mec='red', ecolor='red',
            capsize=5, markersize=8,
            label='_nolegend_', zorder=5)

ax.set_xlabel(r'Redshift $z$', fontsize=14)
ax.set_ylabel(r'Mean ionized fraction $\langle x_e \rangle$', fontsize=14)
title_obj = ax.set_title("Ionization History with Observations", fontsize=14, pad=10)
ax.set_xlim(5, 14)
ax.set_ylim(-0.1, 1.1)
ax.invert_xaxis()
ax.legend(loc='upper left', fontsize=11, frameon=True, framealpha=0.95, edgecolor='black')
ax.tick_params(labelsize=11)

plt.tight_layout()
plt.savefig("xe_vs_z_allHIIeff_total_with_obs_by_technique.png", dpi=300)
title_obj.set_visible(False)
plt.savefig("xe_vs_z_allHIIeff_total_with_obs_by_technique.pdf")
plt.show()

# ----------------------------
# Cell-7 -P-q-total and not perp - ksz power- Multi-HII_EFF_FACTOR PKSZ_TOTAL + <x_e>(z) + tau(z) pipeline (with caching)
# ----------------------------
import os, math
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

sigma_T = const.sigma_T.cgs.value    # Thomson cross-section [cm^2]
c_cms   = const.c.cgs.value          # speed of light [cm/s]
MPC_CM  = 3.0856775814913673e24      # 1 Mpc in cm
T_CMB_K = 2.7255                     # CMB temperature [K]
ne0     = ne0_cgs()                  # mean electron density today [cm^-3]


# user-tunable
HII_EFF_values = [50.0]#[1.0, 10.0, 50.0]   # low to high efficiency
ZS_run = np.array(ZS, dtype=float) # redshift grid to use (same as your notebook)
zsort_asc = np.argsort(ZS_run)   # high -> low ordering used in Eq.(3)
cache_dir = Path("pq_total_cache")  # Changed cache directory name
cache_dir.mkdir(exist_ok=True)

# helper: pretty label and filename fragment from effval
def efflabel_and_frag(effval):
    if effval is None:
        return "HII_EFF_FACTOR=?", "HIIeff_unknown", 0
    exp = int(math.floor(math.log10(effval))) if effval>0 else 0
    coeff = effval / (10**exp) if effval>0 else effval
    if abs(coeff - 1.0) < 1e-3:
        label = fr"$10^{{{exp}}}$"
        frag  = f"1e{exp}"
    else:
        label = fr"${coeff:.1f}\times 10^{{{exp}}}$"
        frag  = f"{coeff:.1f}e{exp}".replace('.', 'p')
    return label, frag, exp

# re-use your interp_loglog if present; else define here
def interp_loglog(xq, xp, fp):
    xp = np.asarray(xp); fp = np.asarray(fp)
    m  = (xp > 0) & (fp > 0)
    lx = np.log(xp[m]); lf = np.log(fp[m])
    # clip query to xp range to avoid extrapolation issues
    lq = np.log(np.clip(xq, xp[m].min(), xp[m].max()))
    return np.exp(np.interp(lq, lx, lf, left=lf[0], right=lf[-1]))

# compute or load per-(z,eff) binned P(k); also cache mean x_e for that z
def compute_or_load_zm_total(z, effval, force=False):
    # cache file names
    _, frag, _ = efflabel_and_frag(effval)
    fname = cache_dir / f"pk_total_z{z}_HIIeff_{frag}.npz"  # Changed filename
    if fname.exists() and not force:
        dat = np.load(fname)
        return dat["k"], dat["P"], float(dat["xe_mean"])
    # not cached: run coeval, compute P(k)
    delta, chi, vx, vy, vz = run_coeval_fields(z, astro_overrides={"HII_EFF_FACTOR": effval})
    # store mean xe for this cube
    xe_mean = float(chi.mean())
    # compute q-total power (uses qtotal_fft_and_power function)
    kbins, Pbins, *_ , eff_used = qtotal_fft_and_power(delta, chi, vx, vy, vz, BOX_LEN, nbins=None, HII_EFF_FACTOR=effval)
    np.savez(fname, k=kbins, P=Pbins, xe_mean=xe_mean, eff=eff_used)
    return kbins, Pbins, xe_mean

# geometry arrays (sorted high->low for integration)
chis = np.array([cosmo.comoving_distance(z).value for z in ZS_run])  # [Mpc] (increasing with z)
a_z   = 1.0/(1.0 + ZS_run)
# reorder high->low
ZS_sorted = ZS_run[zsort_asc]
chi_sorted = chis[zsort_asc]
a_sorted = a_z[zsort_asc]
# dchi (comoving widths) from geometry
dchi_sorted = np.empty_like(chi_sorted)
dchi_sorted[1:] = np.diff(chi_sorted)
dchi_sorted[0] = dchi_sorted[-2] if len(dchi_sorted)>1 else 0.0
dchi_sorted = np.abs(dchi_sorted)
ds_cm_sorted = dchi_sorted * MPC_CM

# ell grid (same as your block)
ells = np.unique(np.round(np.logspace(2.5, 4.2, 60))).astype(int)

# prefactor in Eq.(3)
pref = (sigma_T * ne0 / c_cms)**2

# containers for plotting and summary
Dell_dict = {}      # maps frag -> D_ell array
xe_by_m = {}        # maps frag -> (z_sorted, xe_sorted)
tau_by_m = {}       # maps frag -> (z_sorted, tau_sorted)

# MAIN loop over HII_EFF_values
for effval in HII_EFF_values:
    lab, frag, exp = efflabel_and_frag(effval)
    print(f"\nProcessing HII_EFF_FACTOR = {lab} (TOTAL momentum) ...")
    # collect per-z P(k) and xe means
    k_list = []
    P_list = []
    xe_means = []
    for z in ZS_sorted:                # NOTE: compute in high->low order (same for integration)
        kb, Pb, xe_mean = compute_or_load_zm_total(z, effval, force=False)# force_rerun=True to recompute all
        k_list.append(np.asarray(kb))
        P_list.append(np.asarray(Pb))
        xe_means.append(float(xe_mean))

    # store xe sorted by the same high->low z order
    xe_sorted = np.array(xe_means)
    xe_by_m[frag] = (ZS_sorted.copy(), xe_sorted.copy())

    # compute tau(z) integrated high->low using xe_sorted
    tau = np.zeros_like(ZS_sorted, dtype=float)
    running = 0.0
    for i in range(len(ZS_sorted)-1):
        zmid = 0.5*(ZS_sorted[i] + ZS_sorted[i+1])
        a_mid = 1.0/(1.0 + zmid)
        xe_mid = 0.5*(xe_sorted[i] + xe_sorted[i+1])
        dtaus = sigma_T * ne0 * xe_mid * (a_mid**-2) * (dchi_sorted[i] * MPC_CM)
        running += dtaus
        tau[i+1] = running
    tau_by_m[frag] = (ZS_sorted.copy(), tau.copy())

    # Eq.(3) accumulation: C_ell = pref * Σ_i [ e^{-2τ_i} / (s_i^2 a_i^4) * P(k = ℓ/s_i, z_i) * ds_i ]
    vis2_sorted = np.exp(-2.0 * tau)   # visibility
    C_ell = np.zeros_like(ells, dtype=float)
    for i in range(len(ZS_sorted)):
        s = chi_sorted[i]
        a = a_sorted[i]
        w = vis2_sorted[i] / (s**2 * a**4) * (dchi_sorted[i])
        k_now = ells / s   # ℓ / s [Mpc^-1]
        # interpolate P onto k_now using log-log interpolation (clip within kb range)
        P_now = interp_loglog(k_now, k_list[i], P_list[i])
        C_ell += pref * w * P_now * (MPC_CM**2)

    D_ell_uK2 = ells*(ells+1.0)/(2.0*np.pi) * C_ell * (T_CMB_K**2) * (1e6**2) * 1/2
    Dell_dict[frag] = (ells.copy(), D_ell_uK2.copy())

    dell_fname = cache_dir / f"D_ell_HIIeff_{frag}.txt"
    np.savetxt(dell_fname, np.column_stack([ells, D_ell_uK2]), 
               header="ell   D_ell[uK^2]", 
               fmt=['%d', '%.6e'],
               comments='# ')
    print(f"  Saved D_ell to {dell_fname}")

    # quick print
    D3000 = float(np.interp(3000.0, ells, D_ell_uK2))
    print(f"  D_3000 (μK^2) ≈ {D3000:.4g}  (cached as {frag})")


# ----------------------------
# Professional kSZ Power Spectrum (TOTAL), Ionization, and Optical Depth Plots
# ----------------------------

# ========== kSZ POWER SPECTRUM ==========
fig, ax = plt.subplots(figsize=(8, 6.5))

# Plot your data
for effval in HII_EFF_values:
    lab, frag, exp = efflabel_and_frag(effval)
    ells_plot, Dell = Dell_dict[frag]
    ax.loglog(ells_plot, Dell, lw=2.5, label=f"This work (total)", color='#2E86AB', zorder=10)

# Load and plot FULL Alvarez comparison (no truncation)
try:
    fullsky_data = np.loadtxt("Alvarez2016_binned_smoothed.txt")
    ells_fullsky = fullsky_data[:, 0]
    Dell_fullsky = fullsky_data[:, 1]
    ax.loglog(ells_fullsky, Dell_fullsky, lw=2.0, ls='--', 
              color='black', label='Alvarez et al. (2016)', alpha=0.7, zorder=5)
except FileNotFoundError:
    print("Warning: Alvarez data file not found.")

# Reichardt et al. 2021 measurement at ℓ=3000
ell_reichardt = 3000
D_reichardt = 1.1
D_reichardt_err_upper = 1.0
D_reichardt_err_lower = 0.7

ax.errorbar(ell_reichardt, D_reichardt, 
            yerr=[[D_reichardt_err_lower], [D_reichardt_err_upper]],
            fmt='s', markersize=8, capsize=5, capthick=2, 
            color='red', label='Reichardt et al. (2021)', zorder=15)

ax.set_xlabel(r'Multipole $\ell$', fontsize=14)
ax.set_ylabel(r'$D_\ell$ [$\mu$K$^2$]', fontsize=14)
title_obj = ax.set_title(f"Patchy kSZ Power Spectrum (Total Momentum)\n{BOX_LABEL}", fontsize=14, pad=10)
ax.legend(loc='best', fontsize=12, frameon=True, framealpha=0.95, edgecolor='black')
ax.tick_params(labelsize=11)

plt.tight_layout()
plt.savefig("D_ell_total_overlay_HIIeff_with_fullsky.png", dpi=300)
title_obj.set_visible(False)
plt.savefig("D_ell_total_overlay_HIIeff_with_fullsky.pdf")
plt.show()

# ========== IONIZATION HISTORY ==========
fig, ax = plt.subplots(figsize=(8, 6))

for effval in HII_EFF_values:
    _, frag, _ = efflabel_and_frag(effval)
    zs_plot, xe_plot = xe_by_m[frag]
    order_plot = np.argsort(zs_plot)
    ax.plot(zs_plot[order_plot], xe_plot[order_plot], 
            marker='o', ms=6, lw=2.0, label=f"HII_EFF={efflabel_and_frag(effval)[0]}")

ax.set_xlabel(r'Redshift $z$', fontsize=14)
ax.set_ylabel(r'Mean ionized fraction $\langle x_e \rangle$', fontsize=14)
title_obj = ax.set_title("Ionization History", fontsize=14, pad=10)
ax.invert_xaxis()
ax.legend(loc='best', fontsize=12, frameon=True, framealpha=0.95)
ax.tick_params(labelsize=11)

plt.tight_layout()
plt.savefig("xe_vs_z_allHIIeff_total.png", dpi=300)
title_obj.set_visible(False)
plt.savefig("xe_vs_z_allHIIeff_total.pdf")
plt.show()

# ========== OPTICAL DEPTH ==========
fig, ax = plt.subplots(figsize=(8, 6))

for effval in HII_EFF_values:
    _, frag, _ = efflabel_and_frag(effval)
    zs_plot, tau_plot = tau_by_m[frag]
    order_plot = np.argsort(zs_plot)
    ax.plot(zs_plot[order_plot], tau_plot[order_plot],
            marker='o', ms=6, lw=2.0, label=f"HII_EFF={efflabel_and_frag(effval)[0]}")

ax.set_xlabel(r'Redshift $z$', fontsize=14)
ax.set_ylabel(r'Cumulative optical depth $\tau(z)$', fontsize=14)
title_obj = ax.set_title("Optical Depth Evolution", fontsize=14, pad=10)
ax.invert_xaxis()
ax.legend(loc='best', fontsize=12, frameon=True, framealpha=0.95)
ax.tick_params(labelsize=11)

plt.tight_layout()
plt.savefig("tau_vs_z_allHIIeff_total.png", dpi=300)
title_obj.set_visible(False)
plt.savefig("tau_vs_z_allHIIeff_total.pdf")
plt.show()

print("Done. Cached per-(z,HII_EFF_FACTOR) P_total(k) arrays in", cache_dir.resolve())

# ========== p-q-total vs p-q-per kSZ POWER SPECTRUM COMPARISON (Perp vs Total) ==========
fig, ax = plt.subplots(figsize=(8, 6.5))

# Load and plot perpendicular momentum power spectrum
try:
    perp_data = np.loadtxt("pq_cache/D_ell_HIIeff_5p0e1.txt")
    ells_perp = perp_data[:, 0]
    Dell_perp = perp_data[:, 1]
    ax.loglog(ells_perp, Dell_perp, lw=2.5, 
              label=r"This work ($q_\perp$)", 
              color='#2E86AB', zorder=10)
except FileNotFoundError:
    print("Warning: Perpendicular D_ell cache not found.")

# Load and plot total momentum power spectrum
try:
    total_data = np.loadtxt("pq_total_cache/D_ell_HIIeff_5p0e1.txt")
    ells_total = total_data[:, 0]
    Dell_total = total_data[:, 1]
    ax.loglog(ells_total, Dell_total, lw=2.5, 
              label=r"This work ($q_\mathrm{total}$)", 
              color='#E63946', linestyle='-', zorder=10)
except FileNotFoundError:
    print("Warning: Total D_ell cache not found.")

# Load and plot FULL Alvarez comparison (no truncation)
try:
    fullsky_data = np.loadtxt("Alvarez2016_binned_smoothed.txt")
    ells_fullsky = fullsky_data[:, 0]
    Dell_fullsky = fullsky_data[:, 1]
    ax.loglog(ells_fullsky, Dell_fullsky, lw=2.0, ls='--', 
              color='black', label='Alvarez et al. (2016)', alpha=0.7, zorder=5)
except FileNotFoundError:
    print("Warning: Alvarez data file not found.")

# Reichardt et al. 2021 measurement at ℓ=3000
ell_reichardt = 3000
D_reichardt = 1.1
D_reichardt_err_upper = 1.0
D_reichardt_err_lower = 0.7

ax.errorbar(ell_reichardt, D_reichardt, 
            yerr=[[D_reichardt_err_lower], [D_reichardt_err_upper]],
            fmt='s', markersize=8, capsize=5, capthick=2, 
            color='red', label='Reichardt et al. (2021)', zorder=15)

ax.set_xlabel(r'Multipole $\ell$', fontsize=14)
ax.set_ylabel(r'$D_\ell$ [$\mu$K$^2$]', fontsize=14)
title_obj = ax.set_title(f"Patchy kSZ Power Spectrum Comparison\n{BOX_LABEL}", fontsize=14, pad=10)
ax.legend(loc='best', fontsize=12, frameon=True, framealpha=0.95, edgecolor='black')
plt.tight_layout()
plt.savefig("D_ell_comparison_perp_vs_total.png", dpi=300)
title_obj.set_visible(False)
plt.savefig("D_ell_comparison_perp_vs_total.pdf")
plt.show()

# ----------------------------
# Cell 7-NoLimber - kSZ power WITHOUT Limber approximation (spherical Bessel approach)
# ----------------------------
import os, math
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import spherical_jn

sigma_T = const.sigma_T.cgs.value    # Thomson cross-section [cm^2]
c_cms   = const.c.cgs.value          # speed of light [cm/s]
MPC_CM  = 3.0856775814913673e24      # 1 Mpc in cm
T_CMB_K = 2.7255                     # CMB temperature [K]
ne0     = ne0_cgs()                  # mean electron density today [cm^-3]

# user-tunable
HII_EFF_values = [50.0]
ZS_run = np.array(ZS, dtype=float)
zsort_asc = np.argsort(ZS_run)
cache_dir_nolimber = Path("pq_cache_nolimber")  # DISTINCT cache directory
cache_dir_nolimber.mkdir(exist_ok=True)

# Re-use helper functions
def efflabel_and_frag(effval):
    if effval is None:
        return "HII_EFF_FACTOR=?", "HIIeff_unknown", 0
    exp = int(math.floor(math.log10(effval))) if effval>0 else 0
    coeff = effval / (10**exp) if effval>0 else effval
    if abs(coeff - 1.0) < 1e-3:
        label = fr"$10^{{{exp}}}$"
        frag  = f"1e{exp}"
    else:
        label = fr"${coeff:.1f}\times 10^{{{exp}}}$"
        frag  = f"{coeff:.1f}e{exp}".replace('.', 'p')
    return label, frag, exp

def interp_loglog(xq, xp, fp):
    xp = np.asarray(xp); fp = np.asarray(fp)
    m  = (xp > 0) & (fp > 0)
    lx = np.log(xp[m]); lf = np.log(fp[m])
    lq = np.log(np.clip(xq, xp[m].min(), xp[m].max()))
    return np.exp(np.interp(lq, lx, lf, left=lf[0], right=lf[-1]))

# Load cached P(k) data (re-use from pq_cache)
def load_cached_pk(z, effval):
    _, frag, _ = efflabel_and_frag(effval)
    fname = Path("pq_cache") / f"pk_z{z}_HIIeff_{frag}.npz"
    if fname.exists():
        dat = np.load(fname)
        return dat["k"], dat["P"], float(dat["xe_mean"])
    else:
        raise FileNotFoundError(f"Cached P(k) not found for z={z}. Run Limber calculation first!")

# geometry arrays (sorted high->low for integration)
chis = np.array([cosmo.comoving_distance(z).value for z in ZS_run])  # [Mpc]
a_z   = 1.0/(1.0 + ZS_run)
ZS_sorted = ZS_run[zsort_asc]
chi_sorted = chis[zsort_asc]
a_sorted = a_z[zsort_asc]
dchi_sorted = np.empty_like(chi_sorted)
dchi_sorted[1:] = np.diff(chi_sorted)
dchi_sorted[0] = dchi_sorted[-2] if len(dchi_sorted)>1 else 0.0
dchi_sorted = np.abs(dchi_sorted)

# ell grid - include LOW ell where Limber fails
ells_low = np.arange(2, 100)  # Low ell where Limber breaks down
ells_high = np.unique(np.round(np.logspace(2, 4.2, 40))).astype(int)
ells = np.unique(np.concatenate([ells_low, ells_high]))

# k grid for Bessel integration
k_min = 0.001  # Mpc^-1
k_max = 5.0    # Mpc^-1
n_k = 150
k_grid = np.logspace(np.log10(k_min), np.log10(k_max), n_k)
dk_grid = np.diff(k_grid)
dk_grid = np.append(dk_grid, dk_grid[-1])  # Extend for last point

# prefactor
pref = (sigma_T * ne0 / c_cms)**2

# containers
Dell_dict_nolimber = {}
xe_by_m_nolimber = {}
tau_by_m_nolimber = {}

# MAIN loop
for effval in HII_EFF_values:
    lab, frag, exp = efflabel_and_frag(effval)
    print(f"\nProcessing NO-LIMBER calculation for HII_EFF_FACTOR = {lab} ...")
    
    # Load cached P(k) and xe data
    k_list = []
    P_list = []
    xe_means = []
    for z in ZS_sorted:
        kb, Pb, xe_mean = load_cached_pk(z, effval)
        k_list.append(np.asarray(kb))
        P_list.append(np.asarray(Pb))
        xe_means.append(float(xe_mean))
    
    xe_sorted = np.array(xe_means)
    xe_by_m_nolimber[frag] = (ZS_sorted.copy(), xe_sorted.copy())
    
    # Compute tau(z)
    tau = np.zeros_like(ZS_sorted, dtype=float)
    running = 0.0
    for i in range(len(ZS_sorted)-1):
        zmid = 0.5*(ZS_sorted[i] + ZS_sorted[i+1])
        a_mid = 1.0/(1.0 + zmid)
        xe_mid = 0.5*(xe_sorted[i] + xe_sorted[i+1])
        dtaus = sigma_T * ne0 * xe_mid * (a_mid**-2) * (dchi_sorted[i] * MPC_CM)
        running += dtaus
        tau[i+1] = running
    tau_by_m_nolimber[frag] = (ZS_sorted.copy(), tau.copy())
    
    vis2_sorted = np.exp(-2.0 * tau)
    
    # NO-LIMBER C_ell calculation with spherical Bessel functions
    C_ell = np.zeros_like(ells, dtype=float)
    
    print("  Computing C_ell without Limber approximation...")
    for ell_idx, ell in enumerate(ells):
        if ell_idx % 20 == 0:
            print(f"    ℓ = {ell} ({ell_idx+1}/{len(ells)})")
        
        C_ell_accum = 0.0
        
        # Double loop over redshift shells
        for i in range(len(ZS_sorted)):
            chi_i = chi_sorted[i] * MPC_CM  # Convert to cm
            a_i = a_sorted[i]
            vis_i = np.sqrt(vis2_sorted[i])
            dchi_i = dchi_sorted[i] * MPC_CM  # Convert to cm
            
            # Weight function
            W_i = vis_i / (a_i**2)
            
            for j in range(len(ZS_sorted)):
                chi_j = chi_sorted[j] * MPC_CM  # Convert to cm
                a_j = a_sorted[j]
                vis_j = np.sqrt(vis2_sorted[j])
                dchi_j = dchi_sorted[j] * MPC_CM  # Convert to cm
                
                W_j = vis_j / (a_j**2)
                
                # Use average z for this shell pair
                z_avg_idx = (i + j) // 2
                
                # Interpolate P(k) onto k_grid
                P_k_interp = interp_loglog(k_grid, k_list[z_avg_idx], P_list[z_avg_idx])
                
                # Integrate over k with spherical Bessel functions
                k_integrand = 0.0
                for k_idx in range(len(k_grid)):
                    k = k_grid[k_idx]
                    dk = dk_grid[k_idx]
                    
                    # Spherical Bessel functions j_ℓ(kχ)
                    # Note: chi is in cm, k is in Mpc^-1, so convert k to cm^-1
                    k_cm = k / MPC_CM  # Convert k from Mpc^-1 to cm^-1
                    
                    j_l_i = spherical_jn(ell, k_cm * chi_i)
                    j_l_j = spherical_jn(ell, k_cm * chi_j)
                    
                    k_integrand += k**2 * P_k_interp[k_idx] * j_l_i * j_l_j * dk
                
                C_ell_accum += W_i * W_j * k_integrand * dchi_i * dchi_j
        
        C_ell[ell_idx] = pref * C_ell_accum #* (MPC_CM**2)
    
    # Convert to D_ell
    D_ell_uK2 = ells*(ells+1.0)/(2.0*np.pi) * C_ell * (T_CMB_K**2) * (1e6**2) * 1/2
    Dell_dict_nolimber[frag] = (ells.copy(), D_ell_uK2.copy())
    
    # Save to cache
    dell_fname = cache_dir_nolimber / f"D_ell_HIIeff_{frag}_nolimber.txt"
    np.savetxt(dell_fname, np.column_stack([ells, D_ell_uK2]), 
               header="ell   D_ell[uK^2] (NO LIMBER)", 
               fmt=['%d', '%.6e'],
               comments='# ')
    print(f"  Saved NO-LIMBER D_ell to {dell_fname}")
    
    # Quick print
    D3000 = float(np.interp(3000.0, ells, D_ell_uK2))
    print(f"  D_3000 (μK^2) ≈ {D3000:.4g} [NO LIMBER]")

print("\nDone. NO-LIMBER results cached in", cache_dir_nolimber.resolve())

# ========== kSZ POWER SPECTRUM: LIMBER vs NO-LIMBER COMPARISON ==========
fig, ax = plt.subplots(figsize=(10, 7))

# Load and plot Limber (perpendicular momentum)
try:
    limber_data = np.loadtxt("pq_cache/D_ell_HIIeff_5p0e1.txt")
    ells_limber = limber_data[:, 0]
    Dell_limber = limber_data[:, 1]
    ax.loglog(ells_limber, Dell_limber, lw=2.5, 
              label=r"This work ($q_\perp$, Limber)", 
              color='#2E86AB', zorder=10)
except FileNotFoundError:
    print("Warning: Limber D_ell cache not found.")

# Load and plot No-Limber
try:
    nolimber_data = np.loadtxt("pq_cache_nolimber/D_ell_HIIeff_5p0e1_nolimber.txt")
    ells_nolimber = nolimber_data[:, 0]
    Dell_nolimber = nolimber_data[:, 1]/MPC_CM**2
    ax.loglog(ells_nolimber, Dell_nolimber, lw=2.5, 
              label=r"This work ($q_\perp$, No Limber)", 
              color='#E63946', linestyle='--', zorder=10)
except FileNotFoundError:
    print("Warning: No-Limber D_ell cache not found.")

# Load and plot FULL Alvarez comparison
try:
    fullsky_data = np.loadtxt("Alvarez2016_binned_smoothed.txt")
    ells_fullsky = fullsky_data[:, 0]
    Dell_fullsky = fullsky_data[:, 1]
    ax.loglog(ells_fullsky, Dell_fullsky, lw=2.0, ls=':', 
              color='black', label='Alvarez et al. (2016)', alpha=0.7, zorder=5)
except FileNotFoundError:
    print("Warning: Alvarez data file not found.")

# Reichardt et al. 2021 measurement at ℓ=3000
ell_reichardt = 3000
D_reichardt = 1.1
D_reichardt_err_upper = 1.0
D_reichardt_err_lower = 0.7

ax.errorbar(ell_reichardt, D_reichardt, 
            yerr=[[D_reichardt_err_lower], [D_reichardt_err_upper]],
            fmt='s', markersize=8, capsize=5, capthick=2, 
            color='red', label='Reichardt et al. (2021)', zorder=15)

ax.set_xlabel(r'Multipole $\ell$', fontsize=14)
ax.set_ylabel(r'$D_\ell$ [$\mu$K$^2$]', fontsize=14)
title_obj = ax.set_title(f"kSZ Power: Limber vs No-Limber Approximation\n{BOX_LABEL}", 
                         fontsize=14, pad=10)
ax.legend(loc='best', fontsize=12, frameon=True, framealpha=0.95, edgecolor='black')
ax.tick_params(labelsize=11)
#ax.grid(True, alpha=0.3, linestyle='--', which='both')

plt.tight_layout()
plt.savefig("D_ell_comparison_limber_vs_nolimber.png", dpi=300)
title_obj.set_visible(False)
plt.savefig("D_ell_comparison_limber_vs_nolimber.pdf")
plt.show()

# Print comparison at key ℓ values
print("\nComparison at key multipoles:")
print("ℓ       Limber [μK²]    No-Limber [μK²]    Ratio (NL/L)")
print("-" * 60)
for ell_test in [10, 50, 100, 500, 1000, 3000, 5000]:
    if ell_test <= ells_limber.max() and ell_test <= ells_nolimber.max():
        D_limber_interp = np.interp(ell_test, ells_limber, Dell_limber)
        D_nolimber_interp = np.interp(ell_test, ells_nolimber, Dell_nolimber)
        ratio = D_nolimber_interp / D_limber_interp if D_limber_interp > 0 else np.nan
        print(f"{ell_test:5d}   {D_limber_interp:13.6e}   {D_nolimber_interp:13.6e}   {ratio:8.3f}")